# 🛰️ WallBot Lab
**Wall-following robot navigation with Markov Decision Processes**, plus the Day 5 programs (MDP, ADP, Monte Carlo, Hooke-Jeeves).

This notebook builds the whole project module by module, checks each part as it goes, then launches the Streamlit app.

| Step | What happens |
|---|---|
| 1 | Install packages, create folders, find the dataset |
| 2 | `engine.py`: the robot MDP, checked straight away (value iteration vs policy iteration, CSV output) |
| 3 | `programs.py`: MDP, ADP, Monte Carlo, Hooke-Jeeves, each checked |
| 4 | `web_bridge.py` and the two browser components |
| 5 | Theme, requirements and `app.py` |
| 6 | Launch the app (Jupyter, VS Code or Colab) |

Run all cells from top to bottom.

## 1. Setup

In [ ]:
import importlib.util, os, sys, shutil
from pathlib import Path

need = {"streamlit": "streamlit>=1.32", "plotly": "plotly>=5.18", "pandas": "pandas>=2.0", "numpy": "numpy>=1.24"}
missing = [spec for mod, spec in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    get_ipython().run_line_magic("pip", "install -q " + " ".join(f'"{m}"' for m in missing))
print("Packages ready." if not missing else f"Installed: {', '.join(missing)}")

for folder in ("data", "web", ".streamlit"):
    Path(folder).mkdir(exist_ok=True)
print("Folders ready:", ", ".join(sorted(p.name for p in Path('.').iterdir() if p.is_dir() and not p.name.startswith('__'))))

In [ ]:
IN_COLAB = "google.colab" in sys.modules
target = Path("data") / "sensor_readings_24.csv"
if not target.exists() and Path("sensor_readings_24.csv").exists():
    shutil.copy("sensor_readings_24.csv", target)
if not target.exists() and IN_COLAB:
    from google.colab import files
    print("Upload sensor_readings_24.csv")
    for name, blob in files.upload().items():
        target.write_bytes(blob)
print(f"Dataset: {target} ({target.stat().st_size / 1e6:.2f} MB)" if target.exists()
      else "Dataset not found yet: put sensor_readings_24.csv next to this notebook and re-run this cell.")

## 2. The robot MDP: `engine.py`
Data loading, states (fixed thresholds or quantiles), transition probabilities with Laplace smoothing, shaped rewards, value iteration (the notebook's update plus a threshold) and policy iteration.

In [ ]:
%%writefile engine.py
"""
engine.py - WallBot Lab
=======================
The MDP behind the wall-following robot, with no Streamlit code in it.

    readings  ->  distances  ->  states  ->  P(s'|s,a), R(s,a)  ->  value iteration  ->  V*, pi*

Design choices in this version
    * States use fixed distance thresholds in metres (editable), or quantiles.
    * Transition probabilities are counts with Laplace (additive) smoothing.
    * Rewards are shaped: a smooth Gaussian bonus for holding the target wall
      distance, graded penalties that grow as obstacles get closer, and a small
      bonus for moving forward.
    * Value iteration keeps the Day 5 notebook update and adds a threshold;
      policy iteration is included to cross-check the answer.

Run on its own:  python engine.py   ->  writes optimal_value_function.csv
"""

from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

MOVES = ["Move-Forward", "Slight-Right-Turn", "Sharp-Right-Turn", "Slight-Left-Turn"]
US = [f"US{i}" for i in range(1, 25)]

# Sensor arcs that reproduce the dataset's official 4-sensor file
ARCS = {"SD_front": [11, 12, 13, 14, 15], "SD_left": [18, 19, 20],
        "SD_right": [5, 6, 7, 8, 9], "SD_back": [23, 24]}
DIST = list(ARCS)
SHORT = {"SD_front": "F", "SD_left": "L", "SD_right": "R", "SD_back": "B"}

# Mounting angle of every ultrasound sensor (dataset README), degrees
ANGLE = {1: 180, **{i: -180 + 15 * (i - 1) for i in range(2, 14)}, **{i: 15 * (i - 13) for i in range(14, 25)}}


def locate_dataset(start=None):
    """Find sensor_readings_24.csv next to the code, in data/, or in the working folder."""
    roots = [Path(start).parent if start else Path(__file__).parent, Path.cwd()]
    for root in roots:
        for p in (root / "data" / "sensor_readings_24.csv", root / "sensor_readings_24.csv"):
            if p.exists():
                return p
    return None


def load_readings(source):
    """Read the 24-sensor file (header or not) and add the four simplified distances."""
    raw = pd.read_csv(source, header=None).iloc[:, :25]
    try:
        float(raw.iloc[0, 0])
    except (TypeError, ValueError):
        raw = raw.iloc[1:]
    raw.columns = US + ["Class"]
    raw[US] = raw[US].apply(pd.to_numeric, errors="coerce")
    raw["Class"] = raw["Class"].astype(str).str.strip()
    df = raw.dropna()
    df = df[df["Class"].isin(MOVES)].reset_index(drop=True)
    for name, sensors in ARCS.items():
        df[name] = df[[f"US{i}" for i in sensors]].min(axis=1)
    return df


# ---------------------------------------------------------------------------
# States
# ---------------------------------------------------------------------------

LEVELS = {1: ["Close", "Far"], 2: ["Close", "Medium", "Far"],
          3: ["Close", "Medium", "Far", "Very far"], 4: ["Touching", "Close", "Medium", "Far", "Very far"]}

DEFAULT_CUTS = {"SD_front": [0.7, 1.4], "SD_left": [0.55, 0.9], "SD_right": [0.8, 1.6], "SD_back": [0.8, 1.6]}


def quantile_cuts(df, used, n_levels):
    qs = np.linspace(0, 1, n_levels + 1)[1:-1]
    return {d: sorted(set(np.round(df[d].quantile(qs).values, 2).tolist())) for d in used}


def level_names(cuts):
    return LEVELS[len(cuts)]


def state_code(levels, used, cuts):
    """levels = list of level indices aligned with `used` -> 'F:Close L:Medium ...'"""
    return " ".join(f"{SHORT[d]}:{level_names(cuts[d])[i]}" for d, i in zip(used, levels))


def discretise(df, used, cuts):
    """Vectorised: add a 'State' column plus one level column per distance."""
    out = df.copy()
    lv = np.column_stack([np.digitize(out[d].values, cuts[d], right=False) for d in used])
    for j, d in enumerate(used):
        out[f"lvl_{d}"] = lv[:, j]
    out["State"] = [state_code(row, used, cuts) for row in lv.tolist()]
    return out, lv


# ---------------------------------------------------------------------------
# Rewards (shaped)
# ---------------------------------------------------------------------------

@dataclass
class RewardSpec:
    target_left: float = 0.60     # metres from the wall we want to hold
    tolerance: float = 0.20       # width of the Gaussian bonus
    follow_gain: float = 1.0
    safe_front: float = 0.80      # penalty grows once the front is closer than this
    safe_left: float = 0.45
    crash_gain: float = 4.0
    forward_bonus: float = 0.10   # bonus for Move-Forward
    sharp_cost: float = 0.15      # cost for Sharp-Right-Turn

    def reading_reward(self, front, left):
        follow = self.follow_gain * np.exp(-0.5 * ((left - self.target_left) / max(self.tolerance, 1e-3)) ** 2)
        risk_f = np.clip((self.safe_front - front) / max(self.safe_front, 1e-3), 0, 1)
        risk_l = np.clip((self.safe_left - left) / max(self.safe_left, 1e-3), 0, 1)
        return follow - self.crash_gain * (risk_f + risk_l)

    def action_shaping(self):
        return np.array([self.forward_bonus, 0.0, -self.sharp_cost, 0.0])


# ---------------------------------------------------------------------------
# MDP
# ---------------------------------------------------------------------------

@dataclass
class RobotMDP:
    states: list
    levels: np.ndarray              # (S, n_used) level index per state
    P: np.ndarray                   # (S, A, S)
    R: np.ndarray                   # (S, A)
    counts: np.ndarray              # (S, A, S) raw counts
    seen: np.ndarray                # (S, A) bool - action observed in state
    used: list
    cuts: dict
    extras: dict = field(default_factory=dict)

    @property
    def visits(self):
        return self.counts.sum(axis=(1, 2))

    @property
    def behaviour(self):
        return self.counts.sum(axis=2).argmax(axis=1)


def build_robot_mdp(df, used, cuts, reward: RewardSpec, smoothing=0.5):
    """Estimate P and R from consecutive log rows. Smoothing is spread over the next
    states that were ever reached from s, so impossible jumps stay impossible."""
    dfs, lv = discretise(df, used, cuts)
    states, inv = np.unique(dfs["State"].values, return_inverse=True)
    states = states.tolist()
    S, A = len(states), len(MOVES)
    lvl_of_state = np.zeros((S, len(used)), dtype=int)
    lvl_of_state[inv] = lv
    act = dfs["Class"].map({m: i for i, m in enumerate(MOVES)}).values
    r_next = reward.reading_reward(dfs["SD_front"].values, dfs["SD_left"].values)

    counts = np.zeros((S, A, S))
    rsum = np.zeros((S, A))
    s, a, s2 = inv[:-1], act[:-1], inv[1:]
    np.add.at(counts, (s, a, s2), 1)
    np.add.at(rsum, (s, a), r_next[1:])

    n_sa = counts.sum(axis=2)
    seen = n_sa > 0
    support = counts.sum(axis=1) > 0                 # next states ever reached from s
    P = np.zeros_like(counts)
    for i in range(S):
        sup = support[i]
        for j in range(A):
            if seen[i, j]:
                row = counts[i, j] + smoothing * sup
                P[i, j] = row / row.sum()
    R = np.where(seen, rsum / np.maximum(n_sa, 1), 0.0) + reward.action_shaping()[None, :]

    dead = ~seen.any(axis=1)                         # only seen on the very last row
    for i in np.where(dead)[0]:
        P[i, 0, i], R[i, 0], seen[i, 0] = 1.0, 0.0, True
    mdp = RobotMDP(states, lvl_of_state, P, R, counts, seen, list(used), dict(cuts))
    mdp.extras["data"] = dfs
    mdp.extras["r_next"] = r_next
    return mdp


# ---------------------------------------------------------------------------
# Solvers
# ---------------------------------------------------------------------------

def value_iteration(P, R, gamma, tolerance=1e-6, max_iterations=20_000, seen=None, in_place=False):
    """Notebook update  Q_sa[a] = R[s][a] + gamma * np.dot(T[s][a], V)  with a stopping threshold.
    in_place=True uses each new value immediately (Gauss-Seidel), which usually needs fewer sweeps."""
    S, A = R.shape
    seen = np.ones((S, A), bool) if seen is None else seen
    V = np.zeros(S)
    trace, deltas = [V.copy()], []
    for _ in range(max_iterations):
        V_old = V.copy()
        V_new = V if in_place else np.zeros(S)
        for s in range(S):
            Q_sa = np.full(A, -np.inf)
            for a in range(A):
                if seen[s, a]:
                    Q_sa[a] = R[s][a] + gamma * np.dot(P[s][a], V)
            V_new[s] = np.max(Q_sa)
        V = V_new
        deltas.append(float(np.max(np.abs(V - V_old))))
        trace.append(V.copy())
        if deltas[-1] < tolerance:
            break
    Q = np.where(seen, R + gamma * np.einsum("sat,t->sa", P, V), -np.inf)
    return {"V": V, "Q": Q, "pi": Q.argmax(axis=1), "trace": trace, "deltas": deltas,
            "sweeps": len(deltas), "converged": deltas[-1] < tolerance}


def fast_value_iteration(P, R, gamma, seen, tolerance=1e-8, max_iterations=100_000):
    """Vectorised version used where speed matters (parameter tuning, error baselines)."""
    V = np.zeros(R.shape[0])
    for k in range(max_iterations):
        Q = np.where(seen, R + gamma * np.einsum("sat,t->sa", P, V), -np.inf)
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tolerance:
            V = V_new
            break
        V = V_new
    Q = np.where(seen, R + gamma * np.einsum("sat,t->sa", P, V), -np.inf)
    return V, Q, Q.argmax(axis=1)


def policy_iteration(P, R, gamma, seen, max_rounds=200):
    """Howard's policy iteration: exact evaluation by a linear solve, then greedy improvement."""
    S = R.shape[0]
    pi = np.where(seen, R, -np.inf).argmax(axis=1)
    history = []
    for rnd in range(max_rounds):
        P_pi = P[np.arange(S), pi]
        R_pi = R[np.arange(S), pi]
        V = np.linalg.solve(np.eye(S) - gamma * P_pi, R_pi)
        Q = np.where(seen, R + gamma * np.einsum("sat,t->sa", P, V), -np.inf)
        new_pi = Q.argmax(axis=1)
        changed = int((new_pi != pi).sum())
        history.append({"Round": rnd + 1, "Policy changes": changed, "Mean V": float(V.mean())})
        # keep the old action when it is tied with the best one (avoids cycling)
        keep = np.isclose(Q[np.arange(S), pi], Q.max(axis=1))
        new_pi = np.where(keep, pi, new_pi)
        if (new_pi == pi).all():
            return V, Q, pi, history
        pi = new_pi
    return V, Q, pi, history


# ---------------------------------------------------------------------------
# Output table
# ---------------------------------------------------------------------------

def value_table(mdp: RobotMDP, sol):
    rows = []
    for i, s in enumerate(mdp.states):
        row = {"State": s}
        for d, lvl in zip(mdp.used, mdp.levels[i]):
            row[d] = level_names(mdp.cuts[d])[lvl]
        row["Optimal_Value"] = round(float(sol["V"][i]), 6)
        row["Optimal_Action"] = MOVES[int(sol["pi"][i])]
        for a, m in enumerate(MOVES):
            q = sol["Q"][i, a]
            row[f"Q({m})"] = round(float(q), 6) if np.isfinite(q) else np.nan
        row["Logged_Most_Common"] = MOVES[int(mdp.behaviour[i])]
        row["Visits"] = int(mdp.visits[i])
        rows.append(row)
    return pd.DataFrame(rows).sort_values("Optimal_Value", ascending=False).reset_index(drop=True)


def solve(df, used=DIST, cuts=None, gamma=0.9, tolerance=1e-6, reward=None, smoothing=0.5, in_place=False):
    cuts = cuts or {d: DEFAULT_CUTS[d] for d in used}
    reward = reward or RewardSpec()
    mdp = build_robot_mdp(df, used, cuts, reward, smoothing)
    sol = value_iteration(mdp.P, mdp.R, gamma, tolerance, seen=mdp.seen, in_place=in_place)
    return mdp, sol, value_table(mdp, sol)


if __name__ == "__main__":
    path = locate_dataset(__file__)
    if path is None:
        raise SystemExit("sensor_readings_24.csv not found")
    data = load_readings(path)
    mdp, sol, table = solve(data)
    table.to_csv("optimal_value_function.csv", index=False)
    print(f"{len(mdp.states)} states | {sol['sweeps']} sweeps | converged: {sol['converged']}")
    print(table.head(8).to_string(index=False))
    print("Saved optimal_value_function.csv")


**Check:** build the MDP with the default settings, solve it both ways, and save `optimal_value_function.csv`.

In [ ]:
import importlib, engine
importlib.reload(engine)
data = engine.load_readings(engine.locate_dataset("engine.py"))
mdp, sol, table = engine.solve(data)
V_pi, Q_pi, pi_pi, rounds = engine.policy_iteration(mdp.P, mdp.R, 0.9, mdp.seen)
table.to_csv("optimal_value_function.csv", index=False)
print(f"{len(data):,} readings -> {len(mdp.states)} states")
print(f"Value iteration: {sol['sweeps']} sweeps, converged = {sol['converged']}")
print(f"Policy iteration: {len(rounds)} rounds, max |V_pi - V_vi| = {abs(V_pi - sol['V']).max():.1e}, same policy = {(pi_pi == sol['pi']).all()}")
print("Saved optimal_value_function.csv")
table.head(10)

## 3. The Day 5 programs: `programs.py`

In [ ]:
%%writefile programs.py
"""
programs.py - WallBot Lab
The Day 5 notebook programs, rebuilt as plain Python (no Streamlit):

    1. MDP            - the notebook's 3-state example, value iteration and policy iteration
    2. ADP            - offline Q-learning from the robot's log, and state aggregation
    3. Monte Carlo    - the notebook's grid world: first-visit evaluation and on-policy control
    4. Hooke-Jeeves   - the notebook's pattern search, on test functions and on the robot MDP itself
"""

import numpy as np

import engine as E

# ===========================================================================
# 1. MDP - notebook example
# ===========================================================================

NB_STATES = ["s1", "s2", "s3"]
NB_ACTIONS = ["a1", "a2"]
NB_R = np.array([[5, 10], [2, 3], [8, 1]], dtype=float)
NB_T = np.array([[[0.7, 0.2, 0.1], [0.1, 0.6, 0.3]],
                 [[0.3, 0.4, 0.3], [0.5, 0.3, 0.2]],
                 [[0.4, 0.4, 0.2], [0.2, 0.5, 0.3]]])


def normalise_rows(T):
    """Negative entries become 0; every (s, a) row is rescaled to sum to 1 (uniform if all zero)."""
    T = np.clip(np.nan_to_num(np.asarray(T, dtype=float)), 0, None)
    fixed = []
    for s in range(T.shape[0]):
        for a in range(T.shape[1]):
            tot = T[s, a].sum()
            if tot <= 0:
                T[s, a], _ = 1.0 / T.shape[2], fixed.append((s, a))
            elif abs(tot - 1) > 1e-9:
                T[s, a] = T[s, a] / tot
                fixed.append((s, a))
    return T, fixed


def notebook_value_iteration(T, R, gamma, iterations=1000, tolerance=None):
    """The notebook's loop (fixed number of sweeps), recording V after every sweep."""
    n_s, n_a = R.shape
    V = np.zeros(n_s)
    trace, deltas = [V.copy()], []
    for _ in range(iterations):
        V_new = np.zeros(n_s)
        for s in range(n_s):
            Q_sa = np.zeros(n_a)
            for a in range(n_a):
                Q_sa[a] = R[s][a] + gamma * np.dot(T[s][a], V)
            V_new[s] = np.max(Q_sa)
        deltas.append(float(np.max(np.abs(V_new - V))))
        V = V_new
        trace.append(V.copy())
        if tolerance is not None and deltas[-1] < tolerance:
            break
    Q = R + gamma * np.einsum("sat,t->sa", T, V)
    return V, Q, np.array(trace), deltas


def notebook_policy_iteration(T, R, gamma, max_rounds=50):
    """Policy iteration on the same example; returns the policy and values after each round."""
    n_s = R.shape[0]
    pi = np.zeros(n_s, dtype=int)
    rounds = []
    for k in range(max_rounds):
        V = np.linalg.solve(np.eye(n_s) - gamma * T[np.arange(n_s), pi], R[np.arange(n_s), pi])
        Q = R + gamma * np.einsum("sat,t->sa", T, V)
        rounds.append({"round": k + 1, "policy": pi.copy(), "V": V.copy()})
        new_pi = np.where(np.isclose(Q[np.arange(n_s), pi], Q.max(axis=1)), pi, Q.argmax(axis=1))
        if (new_pi == pi).all():
            break
        pi = new_pi
    return rounds


# ===========================================================================
# 2. ADP
# ===========================================================================

def log_transitions(mdp, reward):
    """(s, a, r, s') tuples straight from consecutive log rows, with the same rewards the MDP uses."""
    dfs = mdp.extras["data"]
    idx = {s: i for i, s in enumerate(mdp.states)}
    s = dfs["State"].map(idx).values
    a = dfs["Class"].map({m: i for i, m in enumerate(E.MOVES)}).values
    r = mdp.extras["r_next"][1:] + reward.action_shaping()[a[:-1]]
    return s[:-1], a[:-1], r, s[1:]


def offline_q_learning(transitions, n_s, n_a, seen, gamma, alpha_mode="decaying", alpha=0.1, epochs=30, seed=0, Q_star=None):
    """Q-learning that replays the logged transitions (no model needed):
         Q(s,a) <- Q(s,a) + alpha * (r + gamma * max_a' Q(s',a') - Q(s,a))
    alpha_mode 'decaying' uses 1 / n(s,a)^0.6, where n(s,a) is how often that pair has been updated."""
    rng = np.random.default_rng(seed)
    s_arr, a_arr, r_arr, s2_arr = transitions
    Q = np.zeros((n_s, n_a))
    n_upd = np.zeros((n_s, n_a))
    mask = np.where(seen, 0.0, -np.inf)
    hist = {"epoch": [], "max_error": [], "mean_error": [], "agreement": []}
    for ep in range(1, epochs + 1):
        order = rng.permutation(len(s_arr))
        for i in order:
            s, a, r, s2 = s_arr[i], a_arr[i], r_arr[i], s2_arr[i]
            n_upd[s, a] += 1
            step = alpha if alpha_mode == "constant" else 1.0 / n_upd[s, a] ** 0.6
            target = r + gamma * np.max(Q[s2] + mask[s2])
            Q[s, a] += step * (target - Q[s, a])
        if Q_star is not None:
            err = np.abs(np.where(seen, Q - Q_star, 0.0))
            hist["epoch"].append(ep)
            hist["max_error"].append(float(err.max()))
            hist["mean_error"].append(float(err[seen].mean()))
            hist["agreement"].append(float(((Q + mask).argmax(1) == (Q_star + mask).argmax(1)).mean()))
    return Q + mask, hist


def state_aggregation(mdp, keep, gamma):
    """Merge states that agree on the kept distances, solve the smaller MDP, and map values back."""
    cols = [mdp.used.index(k) for k in keep]
    keys = [tuple(row[cols]) for row in mdp.levels] if cols else [() for _ in mdp.levels]
    groups = sorted(set(keys))
    gid = np.array([groups.index(k) for k in keys])
    G, A = len(groups), mdp.R.shape[1]
    w = mdp.counts.sum(axis=2)                               # visits of (s, a)
    Pg = np.zeros((G, A, G))
    Rg = np.zeros((G, A))
    seen_g = np.zeros((G, A), dtype=bool)
    for g in range(G):
        members = np.where(gid == g)[0]
        for a in range(A):
            wa = w[members, a] * mdp.seen[members, a]
            if wa.sum() <= 0:
                continue
            seen_g[g, a] = True
            p_full = (wa[:, None] * mdp.P[members, a]).sum(0) / wa.sum()
            np.add.at(Pg[g, a], gid, p_full)
            Rg[g, a] = (wa * mdp.R[members, a]).sum() / wa.sum()
    for g in range(G):
        if not seen_g[g].any():
            Pg[g, 0, g], seen_g[g, 0] = 1.0, True
    Vg, _, _ = E.fast_value_iteration(Pg, Rg, gamma, seen_g)
    V_back = Vg[gid]
    Q_back = np.where(mdp.seen, mdp.R + gamma * np.einsum("sat,t->sa", mdp.P, V_back), -np.inf)
    return {"groups": G, "V": V_back, "pi": Q_back.argmax(1), "group_id": gid}


# ===========================================================================
# 3. Monte Carlo - the notebook's grid world
# ===========================================================================

ARROWS = ["↑", "↓", "←", "→"]            # notebook action order: Up, Down, Left, Right


class GridEnvironment:
    """Notebook environment (start (0,0), goal bottom-right, -1 per step, +10 at the goal), plus optional traps."""

    def __init__(self, grid_size=(4, 4), start_state=(0, 0), goal_state=None, traps=(), max_steps=400):
        self.grid_size = grid_size
        self.start_state = start_state
        self.goal = goal_state or (grid_size[0] - 1, grid_size[1] - 1)
        self.traps = set(map(tuple, traps))
        self.max_steps = max_steps
        self.actions = [0, 1, 2, 3]
        self.reset()

    def reset(self):
        self.current_pos, self.t = self.start_state, 0
        return self.current_pos

    def step(self, action):
        x, y = self.current_pos
        if action == 0: x = max(0, x - 1)
        elif action == 1: x = min(self.grid_size[0] - 1, x + 1)
        elif action == 2: y = max(0, y - 1)
        elif action == 3: y = min(self.grid_size[1] - 1, y + 1)
        self.current_pos, self.t = (x, y), self.t + 1
        if self.current_pos == self.goal:
            return self.current_pos, 10, True, {"end": "goal"}
        if self.current_pos in self.traps:
            return self.current_pos, -10, True, {"end": "trap"}
        if self.t >= self.max_steps:
            return self.current_pos, -1, True, {"end": "timeout"}
        return self.current_pos, -1, False, {"end": None}


def _safe_path_exists(n, traps):
    """Breadth-first search from the start to the goal that never steps on a trap."""
    blocked, goal = set(traps), (n - 1, n - 1)
    frontier, seen = [(0, 0)], {(0, 0)}
    while frontier:
        x, y = frontier.pop()
        if (x, y) == goal:
            return True
        for nx, ny in ((x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)):
            if 0 <= nx < n and 0 <= ny < n and (nx, ny) not in seen and (nx, ny) not in blocked:
                seen.add((nx, ny))
                frontier.append((nx, ny))
    return False


def place_traps(n, count, seed):
    """Random traps that never seal the start off from the goal."""
    rng = np.random.default_rng(seed)
    cells = [(i, j) for i in range(n) for j in range(n) if (i, j) not in ((0, 0), (n - 1, n - 1))]
    count = min(count, len(cells))
    while count > 0:
        for _ in range(300):
            pick = [cells[i] for i in rng.choice(len(cells), size=count, replace=False)]
            if _safe_path_exists(n, pick):
                return pick
        count -= 1                      # too crowded: try with one trap fewer
    return []


def play_episode(env, policy):
    s, steps, end = env.reset(), [], None
    while True:
        a = policy(s)
        s2, r, done, info = env.step(a)
        steps.append((s, a, r))
        s = s2
        if done:
            end = info["end"]
            break
    return steps, s, end


def first_visit_mc(env, policy, episodes, gamma):
    """First-visit Monte Carlo evaluation: V(s) = average return after the first visit to s."""
    n0, n1 = env.grid_size
    total, count = np.zeros((n0, n1)), np.zeros((n0, n1))
    start_trace, returns, lengths, ends = [], [], [], []
    for _ in range(episodes):
        steps, _, end = play_episode(env, policy)
        G, first = 0.0, {}
        rets = [0.0] * len(steps)
        for t in reversed(range(len(steps))):
            G = steps[t][2] + gamma * G
            rets[t] = G
        for t, (s, _, _) in enumerate(steps):
            if s not in first:
                first[s] = rets[t]
        for s, g in first.items():
            total[s] += g
            count[s] += 1
        returns.append(rets[0])
        lengths.append(len(steps))
        ends.append(end)
        start_trace.append(total[env.start_state] / max(count[env.start_state], 1))
    V = np.where(count > 0, total / np.maximum(count, 1), np.nan)
    return V, count, np.array(start_trace), np.array(returns), np.array(lengths), ends


def mc_control(env, episodes, gamma, epsilon, seed, window=50):
    """On-policy first-visit Monte Carlo control with an epsilon-greedy policy."""
    rng = np.random.default_rng(seed)
    n0, n1 = env.grid_size
    Q = np.zeros((n0, n1, 4))
    N = np.zeros((n0, n1, 4))

    def policy(s):
        if rng.random() < epsilon:
            return int(rng.integers(4))
        q = Q[s]
        return int(rng.choice(np.flatnonzero(q == q.max())))

    success, curve = [], []
    for ep in range(1, episodes + 1):
        steps, _, end = play_episode(env, policy)
        G, seen_sa = 0.0, set()
        firsts = {}
        for t, (s, a, _) in enumerate(steps):
            if (s, a) not in firsts:
                firsts[(s, a)] = t
        for t in reversed(range(len(steps))):
            s, a, r = steps[t]
            G = r + gamma * G
            if firsts[(s, a)] == t and (s, a) not in seen_sa:
                seen_sa.add((s, a))
                N[s][a] += 1
                Q[s][a] += (G - Q[s][a]) / N[s][a]
        success.append(end == "goal")
        if ep % window == 0:
            curve.append((ep, float(np.mean(success[-window:]))))
    greedy = Q.argmax(axis=2)
    return Q, greedy, curve


def greedy_rollout(env, greedy, limit=200):
    s, pts = env.reset(), [env.start_state]
    for _ in range(limit):
        s, _, done, info = env.step(int(greedy[s]))
        pts.append(s)
        if done:
            return pts, info["end"]
    return pts, "timeout"


# ===========================================================================
# 4. Hooke-Jeeves
# ===========================================================================

TEST_FUNCTIONS = {
    "x² + y² (notebook)": (lambda v: v[0] ** 2 + v[1] ** 2, (-3, 3, -3, 3), [(0, 0)]),
    "Three-hump camel": (lambda v: 2 * v[0] ** 2 - 1.05 * v[0] ** 4 + v[0] ** 6 / 6 + v[0] * v[1] + v[1] ** 2, (-2.5, 2.5, -2.5, 2.5), [(0, 0)]),
    "Booth": (lambda v: (v[0] + 2 * v[1] - 7) ** 2 + (2 * v[0] + v[1] - 5) ** 2, (-4, 6, -4, 6), [(1, 3)]),
    "Himmelblau": (lambda v: (v[0] ** 2 + v[1] - 11) ** 2 + (v[0] + v[1] ** 2 - 7) ** 2, (-5, 5, -5, 5),
                   [(3, 2), (-2.805118, 3.131312), (-3.779310, -3.283186), (3.584428, -1.848126)]),
    "Rosenbrock": (lambda v: (1 - v[0]) ** 2 + 100 * (v[1] - v[0] ** 2) ** 2, (-2, 2, -1, 3), [(1, 1)]),
}


def hooke_jeeves(func, x0, step_size=0.5, epsilon=1e-6, max_iter=1000, verify_pattern=True):
    """The notebook's algorithm, recording every iteration.
    verify_pattern=False reproduces the notebook exactly (the pattern move is always kept)."""
    x = np.array(x0, dtype=float)
    delta, it, calls = step_size, 0, [0]
    log, probes = [], []

    def f(v):
        calls[0] += 1
        return func(v)

    def explore(x, delta):
        for i in range(len(x)):
            f_val = f(x)
            x[i] += delta
            probes.append(x.copy())
            if f(x) < f_val:
                continue
            x[i] -= 2 * delta
            probes.append(x.copy())
            if f(x) < f_val:
                continue
            x[i] += delta
        return x

    while delta > epsilon and it < max_iter:
        it += 1
        base = np.copy(x)
        x = explore(x, delta)
        if np.array_equal(x, base):
            kind, used = "shrink", delta
            delta /= 2
        else:
            used = delta
            trial = x + (x - base)
            if verify_pattern and not (f(trial) < f(x)):
                kind = "explore"
            else:
                kind, x = "pattern", trial
        log.append({"iter": it, "base": base.copy(), "x": x.copy(), "f": float(func(x)), "step": used, "move": kind})
    return x, log, np.array(probes) if probes else np.zeros((0, len(x0))), calls[0]


class RobotTuner:
    """Objective for tuning the robot MDP with Hooke-Jeeves.
    The data, states and transition model are prepared once; each evaluation only recomputes the
    rewards for a new target wall distance and re-solves for a new discount factor."""

    def __init__(self, df, used, cuts, reward, smoothing):
        self.base = reward
        self.mdp = E.build_robot_mdp(df, used, cuts, reward, smoothing)
        dfs = self.mdp.extras["data"]
        idx = {s: i for i, s in enumerate(self.mdp.states)}
        self.s = dfs["State"].map(idx).values
        self.a = dfs["Class"].map({m: i for i, m in enumerate(E.MOVES)}).values
        self.front, self.left = dfs["SD_front"].values, dfs["SD_left"].values
        self.n_sa = self.mdp.counts.sum(axis=2)
        self.calls = 0

    BOUNDS = ((0.5, 0.99), (0.3, 1.5))

    def clip(self, gamma, target_left):
        return (float(np.clip(gamma, *self.BOUNDS[0])), float(np.clip(target_left, *self.BOUNDS[1])))

    def agreement(self, gamma, target_left):
        gamma, target_left = self.clip(gamma, target_left)
        key = (round(gamma, 4), round(target_left, 4))
        if not hasattr(self, "_memo"):
            self._memo = {}
        if key in self._memo:
            return self._memo[key]
        self.calls += 1
        spec = E.RewardSpec(**{**self.base.__dict__, "target_left": target_left})
        r_next = spec.reading_reward(self.front, self.left)
        S, A = self.mdp.R.shape
        rsum = np.zeros((S, A))
        np.add.at(rsum, (self.s[:-1], self.a[:-1]), r_next[1:])
        R = np.where(self.mdp.seen, rsum / np.maximum(self.n_sa, 1), 0.0) + spec.action_shaping()[None, :]
        _, _, pi = E.fast_value_iteration(self.mdp.P, R, gamma, self.mdp.seen, tolerance=1e-7)
        self._memo[key] = float((pi[self.s] == self.a).mean())
        return self._memo[key]


**Check** each program once:

In [ ]:
import programs
importlib.reload(programs)
V, Q, trace, deltas = programs.notebook_value_iteration(programs.NB_T, programs.NB_R, 0.9)
print("MDP (notebook example): V* =", V.round(4), "policy =", [programs.NB_ACTIONS[a] for a in Q.argmax(1)])

x, log, probes, calls = programs.hooke_jeeves(lambda v: v[0] ** 2 + v[1] ** 2, [1.0, 1.0])
print("Hooke-Jeeves on x^2 + y^2 from (1, 1):", x.round(6), f"({len(log)} iterations)")

env = programs.GridEnvironment((4, 4))
rng = __import__("numpy").random.default_rng(0)
Vmc, *_ = programs.first_visit_mc(env, lambda s: int(rng.integers(4)), 1000, 0.99)
print("Monte Carlo, random policy on the 4x4 grid: V(start) ~", round(float(Vmc[0, 0]), 2))

agg = programs.state_aggregation(mdp, ["SD_front", "SD_left"], 0.9)
print(f"ADP state aggregation: {len(mdp.states)} states -> {agg['groups']}, mean |V - V*| = {abs(agg['V'] - sol['V']).mean():.3f}")

## 4. Browser components
`web_bridge.py` turns results into JSON; `web/simulator.html` is the interactive room and `web/vi_player.html` animates value iteration.

In [ ]:
%%writefile web_bridge.py
"""
web_bridge.py - WallBot Lab
Turns MDP results into JSON for the two browser components in web/ and loads their HTML.
"""

import json
from dataclasses import asdict
from pathlib import Path

import numpy as np

from engine import MOVES, SHORT, level_names

WEB = Path(__file__).parent / "web"

MOVE_COLORS = {"Move-Forward": "#2EC4B6", "Slight-Right-Turn": "#FFB347",
               "Sharp-Right-Turn": "#FF6B6B", "Slight-Left-Turn": "#9D8DF1"}
ARC_COLORS = {"SD_front": "#5AA9E6", "SD_left": "#2EC4B6", "SD_right": "#FFB347", "SD_back": "#9D8DF1"}
SHORT_MOVES = {"Move-Forward": "Forward", "Slight-Right-Turn": "Slight R",
               "Sharp-Right-Turn": "Sharp R", "Slight-Left-Turn": "Slight L"}


def _f(x, nd=4):
    x = float(x)
    return round(x, nd) if np.isfinite(x) else None


def load_html(name, data):
    html = (WEB / name).read_text(encoding="utf-8")
    return html.replace("/*__DATA__*/null", json.dumps(data))


def simulator_data(mdp, sol, reward):
    S = len(mdp.states)
    return {
        "moves": MOVES, "move_colors": MOVE_COLORS, "arc_colors": ARC_COLORS, "short_moves": SHORT_MOVES,
        "used": mdp.used, "short": SHORT,
        "cuts": {d: [float(c) for c in mdp.cuts[d]] for d in mdp.used},
        "names": {d: level_names(mdp.cuts[d]) for d in mdp.used},
        "states": mdp.states,
        "levels": mdp.levels.tolist(),
        "best": {s: MOVES[int(sol["pi"][i])] for i, s in enumerate(mdp.states)},
        "logged": {s: MOVES[int(mdp.behaviour[i])] for i, s in enumerate(mdp.states)},
        "allowed": {s: [MOVES[a] for a in range(len(MOVES)) if mdp.seen[i, a]] for i, s in enumerate(mdp.states)},
        "q": {s: [_f(v, 3) for v in sol["Q"][i]] for i, s in enumerate(mdp.states)},
        "value": {s: _f(sol["V"][i], 3) for i, s in enumerate(mdp.states)},
        "reward": {k: float(v) for k, v in asdict(reward).items()},
        "n_states": S,
    }


def vi_player_data(mdp, sol, gamma, tolerance):
    trace, deltas = sol["trace"], sol["deltas"]
    n = len(trace) - 1
    if n <= 160:
        ks = list(range(n + 1))
    else:
        ks = sorted(set(range(0, 40)) | set(np.unique(np.round(np.geomspace(1, n, 120)).astype(int)).tolist()) | {n})
    frames = [{"k": int(k), "V": [round(float(v), 4) for v in trace[k]], "d": float(deltas[k - 1]) if k else None} for k in ks]
    S, A = mdp.R.shape
    P = [[([[int(j), round(float(mdp.P[s, a, j]), 4)] for j in np.nonzero(mdp.P[s, a])[0]] if mdp.seen[s, a] else [])
          for a in range(A)] for s in range(S)]
    allv = np.concatenate([np.asarray(trace[k]) for k in ks])
    order = np.argsort(-sol["V"]).tolist()
    conv = np.unique(np.linspace(1, n, min(n, 400)).astype(int)).tolist() if n else []
    return {
        "moves": MOVES, "move_colors": MOVE_COLORS, "short_moves": SHORT_MOVES,
        "states": mdp.states, "order": order, "frames": frames,
        "P": P, "R": [[round(float(r), 4) for r in row] for row in mdp.R],
        "seen": mdp.seen.astype(bool).tolist(), "gamma": float(gamma), "tol": float(tolerance),
        "sweeps": int(n), "converged": bool(sol["converged"]),
        "conv": [[int(k), float(deltas[k - 1])] for k in conv],
        "vmin": float(min(0.0, allv.min())), "vmax": float(max(0.0, allv.max())),
        "visits": [int(v) for v in mdp.visits], "final": [round(float(v), 4) for v in sol["V"]],
    }


In [ ]:
%%writefile web/simulator.html
<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<style>
  :root { --bg:#0F1720; --panel:#16212C; --line:#243241; --ink:#E6EDF3; --muted:#8FA1B3;
          --teal:#2EC4B6; --amber:#FFB347; --coral:#FF6B6B; --violet:#9D8DF1; --blue:#5AA9E6; }
  * { box-sizing:border-box; }
  body { margin:0; background:var(--bg); color:var(--ink); font:14px/1.4 "Inter","Segoe UI",system-ui,sans-serif; }
  .wrap { display:grid; grid-template-columns:minmax(320px, 1fr) 300px; gap:14px; padding:6px; }
  @media (max-width: 860px) { .wrap { grid-template-columns:1fr; } }
  .row { display:flex; flex-wrap:wrap; gap:6px; align-items:center; margin-bottom:8px; }
  button, select { font:inherit; color:var(--ink); background:var(--panel); border:1px solid var(--line);
                   border-radius:8px; padding:6px 11px; cursor:pointer; }
  button:hover, select:hover { border-color:var(--teal); }
  button:focus-visible, select:focus-visible, input:focus-visible { outline:2px solid var(--teal); outline-offset:1px; }
  .go { background:var(--teal); color:#08231F; border-color:var(--teal); font-weight:700; min-width:92px; }
  .seg { display:inline-flex; border:1px solid var(--line); border-radius:8px; overflow:hidden; }
  .seg button { border:none; border-radius:0; border-right:1px solid var(--line); }
  .seg button:last-child { border-right:none; }
  .seg button.on { background:#1F3A44; color:var(--teal); font-weight:600; }
  label { color:var(--muted); display:inline-flex; align-items:center; gap:6px; }
  input[type=range] { width:100px; accent-color:var(--teal); }
  canvas { display:block; }
  #room { border-radius:10px; touch-action:none; cursor:crosshair; }
  #trace { border-radius:8px; background:var(--panel); border:1px solid var(--line); margin-top:8px; }
  .card { background:var(--panel); border:1px solid var(--line); border-radius:10px; padding:10px 12px; margin-bottom:10px; }
  .card h5 { margin:0 0 6px; font-size:11.5px; letter-spacing:.06em; text-transform:uppercase; color:var(--muted); font-weight:600; }
  #banner { font-weight:600; }
  #banner.win { color:var(--teal); } #banner.fail { color:var(--coral); }
  #move { font-size:19px; font-weight:700; }
  .kv { display:grid; grid-template-columns:1fr 1fr; gap:6px 10px; }
  .kv div b { display:block; font-size:18px; }
  .kv div span { color:var(--muted); font-size:12px; }
  .bar { display:grid; grid-template-columns:46px 1fr 88px; gap:6px; align-items:center; font-size:12.5px; margin:4px 0; }
  .track { height:8px; background:#0B1219; border-radius:4px; overflow:hidden; }
  .fill { height:100%; border-radius:4px; }
  .muted { color:var(--muted); font-size:12.5px; }
  details summary { color:var(--muted); cursor:pointer; margin:6px 0; }
  .key { display:flex; flex-wrap:wrap; gap:12px; font-size:12px; color:var(--muted); margin-top:6px; }
  .key i { display:inline-block; width:10px; height:10px; border-radius:3px; margin-right:5px; vertical-align:-1px; }
</style></head>
<body>
<div class="wrap">
  <div>
    <div class="row">
      <button id="go" class="go">▶ Run</button>
      <button id="one">Step</button>
      <button id="home">⟲ Restart</button>
      <label>Speed <input id="speed" type="range" min="1" max="40" value="14"></label>
      <select id="map" aria-label="Room"></select>
    </div>
    <div class="row">
      <select id="brain" aria-label="Policy">
        <option value="learned">Learned wall-follower (value iteration on the sensor MDP)</option>
        <option value="grid">Goal seeker (grid MDP, 4 moves with slip)</option>
        <option value="logged">Logged behaviour (what the real robot did)</option>
        <option value="random">Random moves</option>
      </select>
    </div>
    <div class="row">
      <span class="muted">Click the room to place:</span>
      <div class="seg">
        <button data-tool="wall" class="on">▦ Walls</button>
        <button data-tool="start">● Start</button>
        <button data-tool="finish">⚑ Finish</button>
      </div>
      <button id="turnL" title="Rotate start heading 90° left">↺ heading</button>
      <button id="turnR" title="Rotate start heading 90° right">↻ heading</button>
    </div>
    <canvas id="room"></canvas>
    <canvas id="trace"></canvas>
    <div class="key" id="key"></div>
    <details>
      <summary>Uncertainty, motion and view</summary>
      <div class="row">
        <label>Slip <input id="slip" type="range" min="0" max="40" value="0"><span id="slipTxt">0%</span></label>
        <label>Sensor noise <input id="noise" type="range" min="0" max="25" value="0"><span id="noiseTxt">0 cm</span></label>
        <label>Step <input id="stride" type="range" min="4" max="15" value="8"><span id="strideTxt">8 cm</span></label>
      </div>
      <div class="row">
        <label>Slight turn <input id="soft" type="range" min="5" max="40" value="18"><span id="softTxt">18°</span></label>
        <label>Sharp turn <input id="hard" type="range" min="20" max="90" value="45"><span id="hardTxt">45°</span></label>
      </div>
      <div class="row">
        <label><input type="checkbox" id="vRays" checked> sensor rays</label>
        <label><input type="checkbox" id="vPath" checked> path</label>
        <label><input type="checkbox" id="vPlan" checked> grid MDP values and plan</label>
      </div>
    </details>
  </div>
  <div>
    <div class="card"><h5>Mission</h5><div id="banner">Place start and finish, then Run.</div></div>
    <div class="card"><h5>Decision</h5><div id="move">–</div><div id="stateTxt" class="muted"></div></div>
    <div class="card"><h5>Simplified distances</h5><div id="dists"></div></div>
    <div class="card"><h5 id="qHead">Action values</h5><div id="qs"></div></div>
    <div class="card"><h5>Run statistics</h5><div class="kv" id="kv"></div></div>
  </div>
</div>
<script>
(function () {
"use strict";
const D = /*__DATA__*/null;
const MOVES = D.moves, MCOL = D.move_colors, ACOL = D.arc_colors;
const COLS = 20, ROWS = 14, CELL = 0.30, BODY = 0.15, RANGE = 5.0;
const MAX_LEARNED = 2000, MAX_GRID = 800, REACH = 0.35;
const ARC = { SD_front: 0, SD_left: 90, SD_right: -90, SD_back: 180 };
const GRID_MOVES = [[0, -1, "north", "↑"], [1, 0, "east", "→"], [0, 1, "south", "↓"], [-1, 0, "west", "←"]];
const GRID_COL = "#5AA9E6", G_GAMMA = 0.98, G_BUMP = 3;
const el = id => document.getElementById(id);

let walls = [], pos = { x: 2.5, y: 11.5, a: 90 }, shown = { x: 2.5, y: 11.5 };
let start = { c: 2, r: 11, a: 90 }, finish = { c: 16, r: 2 };
let path = [], hist = [], stats = null, now = null, phase = "idle", playing = false, tool = "wall", flash = 0;
let gv = null, gp = null, greach = null, px = 30, slip = 0, noise = 0;
const motion = { stride: 0.08, soft: 18, hard: 45 };
const room = el("room"), g = room.getContext("2d"), trace = el("trace"), tg = trace.getContext("2d");

// ------------------------------------------------------------------ maps
function box(c0, r0, c1, r1) { for (let r = r0; r < r1; r++) for (let c = c0; c < c1; c++) walls[r][c] = 1; }
function empty() { walls = []; for (let r = 0; r < ROWS; r++) { const row = []; for (let c = 0; c < COLS; c++) row.push(r === 0 || c === 0 || r === ROWS - 1 || c === COLS - 1 ? 1 : 0); walls.push(row); } }
const MAPS = {
  "Open hall": { make: () => {}, s: [2, 11, 90], f: [17, 7] },
  "Corridor loop": { make: () => box(6, 5, 14, 9), s: [2, 11, 90], f: [17, 7] },
  "Office with desks": { make: () => { box(6, 5, 9, 7); box(11, 5, 14, 7); box(7, 8, 13, 9); }, s: [2, 11, 90], f: [17, 7] },
  "Warehouse aisles": { make: () => { box(7, 5, 8, 10); box(12, 5, 13, 10); }, s: [2, 11, 90], f: [17, 7] },
  "Maze (goal seeker)": { make: () => { box(4, 1, 5, 9); box(8, 5, 9, 13); box(12, 1, 13, 9); box(16, 5, 17, 13); }, s: [2, 11, 90], f: [18, 12] }
};
Object.keys(MAPS).forEach(k => { const o = document.createElement("option"); o.textContent = k; el("map").appendChild(o); });

const inGrid = (c, r) => c >= 0 && r >= 0 && c < COLS && r < ROWS;
const solid = (c, r) => !inGrid(c, r) || walls[r][c] === 1;
const solidAt = (x, y) => solid(Math.floor(x), Math.floor(y));
function collides(x, y) { const k = BODY / CELL; for (const dx of [-k, 0, k]) for (const dy of [-k, 0, k]) if (solidAt(x + dx, y + dy)) return true; return false; }

// ------------------------------------------------------------------ grid MDP (4 moves, perpendicular slip)
function gridStep(c, r, m) {
  const nc = c + GRID_MOVES[m][0], nr = r + GRID_MOVES[m][1];
  return solid(nc, nr) ? [c, r, true] : [nc, nr, false];
}
function outcomes(m) { return slip > 0 ? [[m, 1 - slip], [(m + 1) % 4, slip / 2], [(m + 3) % 4, slip / 2]] : [[m, 1]]; }
function gridQ(c, r, m) {
  let q = 0;
  for (const [mm, p] of outcomes(m)) {
    const [nc, nr, bump] = gridStep(c, r, mm);
    const v = (nc === finish.c && nr === finish.r) ? 0 : gv[nr][nc];
    q += p * (-1 - (bump ? G_BUMP : 0) + G_GAMMA * v);
  }
  return q;
}
function planGrid() {
  greach = walls.map(row => row.map(() => false));
  const queue = [[finish.c, finish.r]]; greach[finish.r][finish.c] = true;
  while (queue.length) { const [c, r] = queue.shift(); for (let m = 0; m < 4; m++) { const [nc, nr, b] = gridStep(c, r, m); if (!b && !greach[nr][nc]) { greach[nr][nc] = true; queue.push([nc, nr]); } } }
  gv = walls.map(row => row.map(() => 0)); gp = walls.map(row => row.map(() => -1));
  for (let sweep = 0; sweep < 4000; sweep++) {
    let change = 0;
    for (let r = 0; r < ROWS; r++) for (let c = 0; c < COLS; c++) {
      if (walls[r][c] || !greach[r][c] || (c === finish.c && r === finish.r)) continue;
      let best = -Infinity, arg = -1;
      for (let m = 0; m < 4; m++) { const q = gridQ(c, r, m); if (q > best) { best = q; arg = m; } }
      change = Math.max(change, Math.abs(best - gv[r][c])); gv[r][c] = best; gp[r][c] = arg;
    }
    if (change < 1e-6) break;
  }
}

// ------------------------------------------------------------------ sensors and the learned policy
function ray(x, y, deg) {
  const a = deg * Math.PI / 180, dx = Math.cos(a) / CELL, dy = -Math.sin(a) / CELL;
  for (let d = 0.015; d < RANGE; d += 0.015) if (solidAt(x + d * dx, y + d * dy)) return d;
  return RANGE;
}
function gauss() { let u = 0, v = 0; while (!u) u = Math.random(); while (!v) v = Math.random(); return Math.sqrt(-2 * Math.log(u)) * Math.cos(2 * Math.PI * v); }
function scan() {
  const dist = {}, rays = {};
  for (const k of Object.keys(ARC)) {
    let low = RANGE; const list = [];
    for (let o = -30; o <= 30; o += 5) { const deg = pos.a + ARC[k] + o, d = ray(pos.x, pos.y, deg); list.push([deg, d]); low = Math.min(low, d); }
    if (noise > 0) low = Math.min(RANGE, Math.max(0.05, low + gauss() * noise));
    dist[k] = low; rays[k] = list;
  }
  return { dist: dist, rays: rays };
}
function levelOf(k, v) { const cuts = D.cuts[k]; let i = 0; while (i < cuts.length && v >= cuts[i]) i++; return i; }
function codeOf(lv) { return D.used.map((k, j) => D.short[k] + ":" + D.names[k][lv[j]]).join(" "); }
function closestKnown(lv) {
  let best = null, gap = Infinity;
  D.states.forEach((s, i) => { const L = D.levels[i]; let d = 0; for (let j = 0; j < lv.length; j++) d += Math.abs(L[j] - lv[j]); if (d < gap) { gap = d; best = s; } });
  return best;
}
function shapedReward(dd) {
  const R = D.reward;
  const follow = R.follow_gain * Math.exp(-0.5 * Math.pow((dd.SD_left - R.target_left) / Math.max(R.tolerance, 1e-3), 2));
  const rf = Math.min(1, Math.max(0, (R.safe_front - dd.SD_front) / R.safe_front));
  const rl = Math.min(1, Math.max(0, (R.safe_left - dd.SD_left) / R.safe_left));
  return follow - R.crash_gain * (rf + rl);
}
const gridMode = () => el("brain").value === "grid";
function think() {
  const sc = scan();
  if (gridMode()) {
    const c = Math.floor(pos.x), r = Math.floor(pos.y), atF = c === finish.c && r === finish.r;
    const m = (gp && inGrid(c, r) && !atF) ? gp[r][c] : -1;
    now = { sc: sc, grid: true, c: c, r: r, m: m, label: "square (" + c + ", " + r + ")",
            move: atF ? "At the finish" : (m >= 0 ? GRID_MOVES[m][3] + " go " + GRID_MOVES[m][2] : "No route") };
    return;
  }
  const lv = D.used.map(k => levelOf(k, sc.dist[k]));
  let label = codeOf(lv), guessed = false;
  if (!(label in D.best)) { label = closestKnown(lv); guessed = true; }
  const b = el("brain").value; let mv;
  if (b === "logged") mv = D.logged[label];
  else if (b === "random") { const opts = D.allowed[label]; mv = opts[Math.floor(Math.random() * opts.length)]; }
  else mv = D.best[label];
  now = { sc: sc, grid: false, label: label, guessed: guessed, move: mv };
}

// ------------------------------------------------------------------ stepping
function fresh() { return { steps: 0, metres: 0, bumps: 0, slips: 0, reward: 0, holding: 0, guessed: 0 }; }
const limit = () => gridMode() ? MAX_GRID : MAX_LEARNED;
const over = () => phase === "won" || phase === "timeout" || phase === "blocked";
const atFinish = () => Math.hypot(pos.x - finish.c - 0.5, pos.y - finish.r - 0.5) * CELL <= REACH;
function end(p) { phase = p; setPlaying(false); panel(); }
function advance() {
  if (over()) return;
  if (atFinish()) { end("won"); return; }
  if (!now) think();
  if (now.grid) {
    if (now.m < 0) { end("blocked"); return; }
    let m = now.m, slipped = false;
    if (Math.random() < slip) { m = (m + (Math.random() < 0.5 ? 1 : 3)) % 4; slipped = true; }
    const [nc, nr, bump] = gridStep(now.c, now.r, m);
    pos.a = [90, 0, 270, 180][m];
    if (bump) { stats.bumps++; flash = 10; } else { pos.x = nc + 0.5; pos.y = nr + 0.5; stats.metres += CELL; }
    if (slipped) stats.slips++;
    path.push([pos.x, pos.y, GRID_COL]);
  } else {
    let mv = now.move;
    if (Math.random() < slip) { mv = MOVES[Math.floor(Math.random() * 4)]; stats.slips++; }
    if (now.guessed) stats.guessed++;
    let adv = motion.stride;
    if (mv === "Slight-Right-Turn") pos.a -= motion.soft;
    else if (mv === "Sharp-Right-Turn") { pos.a -= motion.hard; adv *= 0.25; }
    else if (mv === "Slight-Left-Turn") pos.a += motion.soft;
    pos.a = ((pos.a % 360) + 360) % 360;
    const rad = pos.a * Math.PI / 180, nx = pos.x + adv * Math.cos(rad) / CELL, ny = pos.y - adv * Math.sin(rad) / CELL;
    if (collides(nx, ny)) { stats.bumps++; flash = 10; } else { pos.x = nx; pos.y = ny; stats.metres += adv; }
    shown.x = pos.x; shown.y = pos.y;
    path.push([pos.x, pos.y, MCOL[mv]]);
  }
  if (path.length > 4000) path.shift();
  stats.steps++;
  think();
  const dd = now.sc.dist;
  stats.reward += shapedReward(dd);
  if (Math.abs(dd.SD_left - D.reward.target_left) <= D.reward.tolerance) stats.holding++;
  hist.push([dd.SD_left, dd.SD_front]); if (hist.length > 240) hist.shift();
  if (atFinish()) end("won");
  else if (stats.steps >= limit()) end("timeout");
  else phase = "running";
}

// ------------------------------------------------------------------ drawing
function fit() {
  const w = room.parentElement.clientWidth || 640;
  px = Math.max(14, Math.floor(Math.min(w, 860) / COLS));
  const dpr = window.devicePixelRatio || 1;
  room.width = COLS * px * dpr; room.height = ROWS * px * dpr; room.style.width = COLS * px + "px"; room.style.height = ROWS * px + "px";
  g.setTransform(dpr, 0, 0, dpr, 0, 0);
  trace.width = COLS * px * dpr; trace.height = 90 * dpr; trace.style.width = COLS * px + "px"; trace.style.height = "90px";
  tg.setTransform(dpr, 0, 0, dpr, 0, 0);
  draw();
}
function draw() {
  const W = COLS * px, H = ROWS * px;
  g.fillStyle = "#101A24"; g.fillRect(0, 0, W, H);
  const plan = gridMode() && el("vPlan").checked && gv;
  let vlo = 0;
  if (plan) for (let r = 0; r < ROWS; r++) for (let c = 0; c < COLS; c++) if (!walls[r][c] && greach[r][c]) vlo = Math.min(vlo, gv[r][c]);
  for (let r = 0; r < ROWS; r++) for (let c = 0; c < COLS; c++) {
    if (walls[r][c]) { g.fillStyle = "#3A4B5C"; g.fillRect(c * px, r * px, px, px); g.fillStyle = "#4A5D70"; g.fillRect(c * px, r * px, px, 3); continue; }
    if (plan) { g.fillStyle = greach[r][c] ? "rgba(90,169,230," + (0.05 + 0.35 * (vlo < 0 ? 1 - gv[r][c] / vlo : 0)).toFixed(3) + ")" : "rgba(255,107,107,0.10)"; g.fillRect(c * px, r * px, px, px); }
  }
  g.strokeStyle = "rgba(143,161,179,0.10)"; g.lineWidth = 1; g.beginPath();
  for (let c = 0; c <= COLS; c++) { g.moveTo(c * px + 0.5, 0); g.lineTo(c * px + 0.5, H); }
  for (let r = 0; r <= ROWS; r++) { g.moveTo(0, r * px + 0.5); g.lineTo(W, r * px + 0.5); }
  g.stroke();
  if (plan) {
    if (px >= 20) {
      g.fillStyle = "rgba(230,237,243,0.35)"; g.font = Math.round(px * 0.45) + "px system-ui"; g.textAlign = "center"; g.textBaseline = "middle";
      for (let r = 0; r < ROWS; r++) for (let c = 0; c < COLS; c++) if (!walls[r][c] && gp[r][c] >= 0) g.fillText(GRID_MOVES[gp[r][c]][3], (c + 0.5) * px, (r + 0.5) * px);
    }
    let c = Math.floor(pos.x), r = Math.floor(pos.y), n = 0;
    if (inGrid(c, r) && greach[r][c]) {
      g.setLineDash([5, 5]); g.strokeStyle = "#5AA9E6"; g.lineWidth = 2.5; g.beginPath(); g.moveTo((c + .5) * px, (r + .5) * px);
      while (!(c === finish.c && r === finish.r) && n < 300) { const m = gp[r][c]; if (m < 0) break; const [nc, nr, b] = gridStep(c, r, m); if (b) break; c = nc; r = nr; n++; g.lineTo((c + .5) * px, (r + .5) * px); }
      g.stroke(); g.setLineDash([]);
    }
  }
  if (el("vPath").checked && path.length > 1) {
    g.lineWidth = 2; g.lineCap = "round";
    for (let i = 1; i < path.length; i++) { g.globalAlpha = 0.2 + 0.8 * i / path.length; g.strokeStyle = path[i][2]; g.beginPath(); g.moveTo(path[i - 1][0] * px, path[i - 1][1] * px); g.lineTo(path[i][0] * px, path[i][1] * px); g.stroke(); }
    g.globalAlpha = 1;
  }
  // start pad
  const sx = (start.c + .5) * px, sy = (start.r + .5) * px;
  g.fillStyle = "rgba(46,196,182,0.18)"; g.strokeStyle = "#2EC4B6"; g.lineWidth = 2;
  g.beginPath(); g.arc(sx, sy, px * 0.42, 0, 2 * Math.PI); g.fill(); g.stroke();
  const sa = start.a * Math.PI / 180;
  g.beginPath(); g.moveTo(sx + Math.cos(sa) * px * 0.42, sy - Math.sin(sa) * px * 0.42); g.lineTo(sx + Math.cos(sa) * px * 0.62, sy - Math.sin(sa) * px * 0.62); g.stroke();
  // finish flag
  const fx = finish.c * px, fy = finish.r * px;
  g.fillStyle = "rgba(255,179,71,0.16)"; g.fillRect(fx, fy, px, px); g.strokeStyle = "#FFB347"; g.lineWidth = 2; g.strokeRect(fx + 1.5, fy + 1.5, px - 3, px - 3);
  g.beginPath(); g.moveTo(fx + px * 0.3, fy + px * 0.85); g.lineTo(fx + px * 0.3, fy + px * 0.15); g.stroke();
  g.fillStyle = "#FFB347"; g.beginPath(); g.moveTo(fx + px * 0.3, fy + px * 0.15); g.lineTo(fx + px * 0.8, fy + px * 0.3); g.lineTo(fx + px * 0.3, fy + px * 0.45); g.fill();
  // rover
  const ease = now && now.grid ? 0.3 : 1;
  shown.x += (pos.x - shown.x) * ease; shown.y += (pos.y - shown.y) * ease;
  const rx = shown.x * px, ry = shown.y * px;
  if (now && el("vRays").checked && !plan) {
    for (const k of Object.keys(ARC)) {
      g.strokeStyle = ACOL[k]; g.lineWidth = 1;
      now.sc.rays[k].forEach(([deg, d], i) => {
        if (i % 3 !== 0) return;
        const a = deg * Math.PI / 180; g.globalAlpha = 0.45;
        g.beginPath(); g.moveTo(rx, ry); g.lineTo(rx + Math.cos(a) * d / CELL * px, ry - Math.sin(a) * d / CELL * px); g.stroke();
      });
      g.globalAlpha = 1;
      const best = now.sc.rays[k].reduce((a, b) => (b[1] < a[1] ? b : a));
      const a = best[0] * Math.PI / 180;
      g.fillStyle = ACOL[k]; g.beginPath(); g.arc(rx + Math.cos(a) * best[1] / CELL * px, ry - Math.sin(a) * best[1] / CELL * px, 3.5, 0, 2 * Math.PI); g.fill();
    }
  }
  const body = BODY / CELL * px, rad = pos.a * Math.PI / 180;
  g.save(); g.translate(rx, ry); g.rotate(-rad);
  if (flash > 0) { g.strokeStyle = "rgba(255,107,107," + flash / 10 + ")"; g.lineWidth = 3; g.beginPath(); g.arc(0, 0, body + 6, 0, 2 * Math.PI); g.stroke(); }
  if (phase === "won") { g.strokeStyle = "#2EC4B6"; g.lineWidth = 3; g.beginPath(); g.arc(0, 0, body + 9, 0, 2 * Math.PI); g.stroke(); }
  g.fillStyle = now ? (now.grid ? GRID_COL : (MCOL[now.move] || "#2EC4B6")) : "#2EC4B6";
  const bw = body * 1.8, bh = body * 1.5, rr = 4;
  g.beginPath(); g.moveTo(-bw / 2 + rr, -bh / 2); g.lineTo(bw / 2 - rr, -bh / 2); g.quadraticCurveTo(bw / 2, -bh / 2, bw / 2, -bh / 2 + rr);
  g.lineTo(bw / 2, bh / 2 - rr); g.quadraticCurveTo(bw / 2, bh / 2, bw / 2 - rr, bh / 2); g.lineTo(-bw / 2 + rr, bh / 2);
  g.quadraticCurveTo(-bw / 2, bh / 2, -bw / 2, bh / 2 - rr); g.lineTo(-bw / 2, -bh / 2 + rr); g.quadraticCurveTo(-bw / 2, -bh / 2, -bw / 2 + rr, -bh / 2); g.fill();
  g.fillStyle = "#0F1720"; g.beginPath(); g.moveTo(bw / 2 - 2, 0); g.lineTo(bw / 2 - bh * 0.55, -bh * 0.28); g.lineTo(bw / 2 - bh * 0.55, bh * 0.28); g.fill();
  g.restore();
  if (over()) {
    const txt = phase === "won" ? "Finish reached in " + stats.steps + " steps" : phase === "timeout" ? "No finish after " + stats.steps + " steps" : "No route to the finish";
    g.font = "600 15px system-ui"; const tw = g.measureText(txt).width + 30;
    g.fillStyle = phase === "won" ? "rgba(46,196,182,0.95)" : "rgba(255,107,107,0.95)"; g.fillRect((W - tw) / 2, 10, tw, 32);
    g.fillStyle = "#08231F"; g.textAlign = "center"; g.textBaseline = "middle"; g.fillText(txt, W / 2, 26);
  }
  drawTrace();
}
function drawTrace() {
  const W = COLS * px, H = 90, R = D.reward, top = 2.0;
  tg.clearRect(0, 0, W, H);
  const y = v => H - 14 - (H - 24) * Math.min(v, top) / top;
  tg.fillStyle = "rgba(46,196,182,0.14)"; tg.fillRect(34, y(R.target_left + R.tolerance), W - 40, y(R.target_left - R.tolerance) - y(R.target_left + R.tolerance));
  tg.fillStyle = "#8FA1B3"; tg.font = "10px system-ui"; tg.textAlign = "right"; tg.textBaseline = "middle";
  [0, 1, 2].forEach(v => tg.fillText(v + " m", 30, y(v)));
  tg.textAlign = "left"; tg.fillText("left distance (target band shaded) and front distance, last 240 steps", 38, 8);
  const n = hist.length; if (n < 2) return;
  const xAt = i => 34 + (W - 40) * i / 239;
  [[0, "#2EC4B6"], [1, "#5AA9E6"]].forEach(([k, col]) => {
    tg.strokeStyle = col; tg.lineWidth = 1.6; tg.beginPath();
    hist.forEach((h, i) => { const X = xAt(i + 240 - n), Y = y(h[k]); if (i) tg.lineTo(X, Y); else tg.moveTo(X, Y); }); tg.stroke();
  });
}

// ------------------------------------------------------------------ side panel
const fx = (v, d) => (v === null || v === undefined || !isFinite(v)) ? "–" : Number(v).toFixed(d);
function panel() {
  if (!now || !stats) return;
  const b = el("banner");
  if (phase === "won") { b.className = "win"; b.textContent = "Finish reached: " + stats.steps + " steps, " + stats.metres.toFixed(1) + " m, " + stats.bumps + " bumps."; }
  else if (phase === "timeout") { b.className = "fail"; b.textContent = "Stopped after " + stats.steps + " steps. " + (gridMode() ? "Try less slip." : "The wall-follower only passes finishes near the wall it follows. Move the flag next to a wall, or switch to the goal seeker."); }
  else if (phase === "blocked") { b.className = "fail"; b.textContent = "The finish is walled off. Erase a wall or move the flag."; }
  else { b.className = ""; b.textContent = playing ? "Driving… step " + stats.steps : (stats.steps ? "Paused at step " + stats.steps + "." : "Ready. Press Run."); }
  el("move").textContent = now.move; el("move").style.color = now.grid ? GRID_COL : (MCOL[now.move] || "#E6EDF3");
  if (now.grid) {
    const v = (gv && inGrid(now.c, now.r)) ? gv[now.r][now.c] : null;
    el("stateTxt").innerHTML = "State " + now.label + ". V = " + fx(v, 2) + ", about " + (v === null ? "–" : Math.max(0, -v).toFixed(0)) + " moves to go.";
  } else {
    el("stateTxt").innerHTML = "State " + now.label + (now.guessed ? " <span style='color:#FF6B6B'>(closest known state, this exact reading is not in the log)</span>" : "") + ". V* = " + fx(D.value[now.label], 2);
  }
  el("dists").innerHTML = Object.keys(ARC).map(k => {
    const v = now.sc.dist[k], used = D.used.indexOf(k) >= 0, lvl = used ? D.names[k][levelOf(k, v)] : "";
    return "<div class='bar'><span>" + k.slice(3) + "</span><div class='track'><div class='fill' style='width:" + Math.min(100, v / RANGE * 100).toFixed(1) +
      "%;background:" + ACOL[k] + "'></div></div><span>" + v.toFixed(2) + " m" + (lvl ? ", " + lvl : "") + "</span></div>";
  }).join("");
  let html = "";
  if (now.grid) {
    el("qHead").textContent = "Grid MDP action values";
    if (gv && inGrid(now.c, now.r) && greach[now.r][now.c] && now.m >= 0) {
      const q = [0, 1, 2, 3].map(m => [gridQ(now.c, now.r, m), m]).sort((a, b) => b[0] - a[0]);
      const lo = q[3][0], hi = q[0][0];
      q.forEach(([v, m]) => { html += "<div class='bar'><span>" + GRID_MOVES[m][3] + "</span><div class='track'><div class='fill' style='width:" + (hi > lo ? 10 + 90 * (v - lo) / (hi - lo) : 100).toFixed(0) + "%;background:" + GRID_COL + "'></div></div><span>" + v.toFixed(2) + (m === now.m ? " ★" : "") + "</span></div>"; });
    } else html = "<span class='muted'>Nothing to rank here.</span>";
  } else {
    el("qHead").textContent = "Sensor MDP action values Q(s, a)";
    const q = D.q[now.label] || [], ok = q.filter(v => v !== null), lo = ok.length ? Math.min.apply(null, ok) : 0, hi = ok.length ? Math.max.apply(null, ok) : 1;
    MOVES.forEach((m, i) => {
      const v = q[i], has = v !== null && v !== undefined, star = m === D.best[now.label];
      html += "<div class='bar'><span style='font-size:11px'>" + D.short_moves[m] + "</span><div class='track'><div class='fill' style='width:" + (has ? (hi > lo ? 10 + 90 * (v - lo) / (hi - lo) : 100) : 0).toFixed(0) +
        "%;background:" + MCOL[m] + "'></div></div><span>" + (has ? v.toFixed(2) : "not in log") + (star ? " ★" : "") + "</span></div>";
    });
  }
  el("qs").innerHTML = html;
  el("kv").innerHTML =
    "<div><b>" + stats.steps + "</b><span>steps</span></div><div><b>" + stats.metres.toFixed(1) + " m</b><span>travelled</span></div>" +
    "<div><b>" + stats.bumps + "</b><span>bumps</span></div><div><b>" + stats.slips + "</b><span>slipped moves</span></div>" +
    "<div><b>" + (stats.steps ? (100 * stats.holding / stats.steps).toFixed(0) : 0) + "%</b><span>in target band</span></div>" +
    "<div><b>" + (stats.steps ? (stats.reward / stats.steps).toFixed(2) : "0.00") + "</b><span>mean reward per step</span></div>";
}

// ------------------------------------------------------------------ controls
function restart() {
  pos = { x: start.c + 0.5, y: start.r + 0.5, a: start.a }; shown = { x: pos.x, y: pos.y };
  path = []; hist = []; stats = fresh(); phase = "idle"; planGrid(); think(); draw(); panel();
}
function loadMap() { empty(); const M = MAPS[el("map").value]; M.make(); start = { c: M.s[0], r: M.s[1], a: M.s[2] }; finish = { c: M.f[0], r: M.f[1] }; restart(); }
function setPlaying(v) { playing = v; el("go").textContent = v ? "⏸ Pause" : "▶ Run"; }
el("go").onclick = () => { if (!playing && over()) restart(); setPlaying(!playing); panel(); };
el("one").onclick = () => { setPlaying(false); advance(); draw(); panel(); };
el("home").onclick = () => { setPlaying(false); restart(); };
el("map").onchange = () => { setPlaying(false); loadMap(); };
el("brain").onchange = () => {
  if (gridMode()) { pos.x = Math.floor(pos.x) + 0.5; pos.y = Math.floor(pos.y) + 0.5; shown.x = pos.x; shown.y = pos.y; }
  if (!over()) phase = stats.steps ? "paused" : "idle"; think(); draw(); panel();
};
document.querySelectorAll(".seg button").forEach(btn => btn.onclick = () => {
  tool = btn.dataset.tool; document.querySelectorAll(".seg button").forEach(x => x.classList.toggle("on", x === btn));
});
el("turnL").onclick = () => { start.a = (start.a + 90) % 360; if (!stats.steps) restart(); else draw(); };
el("turnR").onclick = () => { start.a = (start.a + 270) % 360; if (!stats.steps) restart(); else draw(); };
function slider(id, out, fn, after) { const s = el(id); s.oninput = () => { el(out).textContent = fn(+s.value); if (after) after(); }; el(out).textContent = fn(+s.value); }
slider("slip", "slipTxt", v => { slip = v / 100; return v + "%"; }, () => { planGrid(); think(); draw(); panel(); });
slider("noise", "noiseTxt", v => { noise = v / 100; return v + " cm"; });
slider("stride", "strideTxt", v => { motion.stride = v / 100; return v + " cm"; });
slider("soft", "softTxt", v => { motion.soft = v; return v + "°"; });
slider("hard", "hardTxt", v => { motion.hard = v; return v + "°"; });
["vRays", "vPath", "vPlan"].forEach(id => el(id).onchange = draw);

let paint = null;
function cellAt(e) { const b = room.getBoundingClientRect(); return [Math.floor((e.clientX - b.left) / px), Math.floor((e.clientY - b.top) / px)]; }
const inner = (c, r) => c > 0 && r > 0 && c < COLS - 1 && r < ROWS - 1;
function paintCell(c, r) {
  if (!inner(c, r)) return;
  if (paint === 1 && ((c === finish.c && r === finish.r) || (c === start.c && r === start.r) || (c === Math.floor(pos.x) && r === Math.floor(pos.y)))) return;
  walls[r][c] = paint;
}
room.addEventListener("pointerdown", e => {
  const [c, r] = cellAt(e); if (!inner(c, r)) return;
  if (tool === "start") { if (!walls[r][c]) { start.c = c; start.r = r; setPlaying(false); restart(); } return; }
  if (tool === "finish") { if (!walls[r][c]) { finish.c = c; finish.r = r; planGrid(); if (over()) phase = "paused"; think(); draw(); panel(); } return; }
  paint = walls[r][c] ? 0 : 1; paintCell(c, r);
  try { room.setPointerCapture(e.pointerId); } catch (err) { /* ignore */ }
  draw();
});
room.addEventListener("pointermove", e => { if (paint === null) return; const [c, r] = cellAt(e); paintCell(c, r); draw(); });
window.addEventListener("pointerup", () => { if (paint !== null) { paint = null; planGrid(); if (phase === "blocked") phase = "paused"; think(); draw(); panel(); } });

el("key").innerHTML = MOVES.map(m => "<span><i style='background:" + MCOL[m] + "'></i>" + m + "</span>").join("") + "<span><i style='background:" + GRID_COL + "'></i>Grid move</span>";

window.__wallbot = { run: n => { let i = 0; while (i < n && !over()) { advance(); i++; } panel(); return { phase: phase, steps: stats.steps, bumps: stats.bumps }; },
                     visits: () => path.map(p => [Math.floor(p[0]), Math.floor(p[1])]) };
let acc = 0, last = null;
function frame(t) {
  if (last === null) last = t;
  const dt = Math.min(0.1, (t - last) / 1000); last = t;
  if (playing) {
    acc += dt * +el("speed").value * (gridMode() ? 0.4 : 1);
    let n = 0, moved = false;
    while (acc >= 1 && n < 40 && playing) { advance(); acc -= 1; n++; moved = true; }
    if (moved) panel();
  } else acc = 0;
  if (flash > 0) flash--;
  draw(); requestAnimationFrame(frame);
}
window.addEventListener("resize", fit);
loadMap(); fit(); requestAnimationFrame(frame);
})();
</script>
</body></html>


In [ ]:
%%writefile web/vi_player.html
<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<style>
  :root { --bg:#0F1720; --panel:#16212C; --line:#243241; --ink:#E6EDF3; --muted:#8FA1B3; --teal:#2EC4B6; --amber:#FFB347; --coral:#FF6B6B; }
  * { box-sizing:border-box; }
  body { margin:0; background:var(--bg); color:var(--ink); font:14px/1.4 "Inter","Segoe UI",system-ui,sans-serif; }
  .wrap { display:grid; grid-template-columns:minmax(320px, 1fr) 340px; gap:14px; padding:6px; }
  @media (max-width: 900px) { .wrap { grid-template-columns:1fr; } }
  .row { display:flex; flex-wrap:wrap; gap:6px; align-items:center; margin-bottom:8px; }
  button, select { font:inherit; color:var(--ink); background:var(--panel); border:1px solid var(--line); border-radius:8px; padding:6px 11px; cursor:pointer; }
  button:hover { border-color:var(--teal); }
  button:focus-visible, select:focus-visible, input:focus-visible { outline:2px solid var(--teal); outline-offset:1px; }
  .go { background:var(--teal); color:#08231F; border-color:var(--teal); font-weight:700; min-width:92px; }
  input[type=range] { flex:1; min-width:140px; accent-color:var(--teal); }
  .stats { display:flex; gap:22px; margin-bottom:8px; flex-wrap:wrap; }
  .stats div span { display:block; color:var(--muted); font-size:11.5px; text-transform:uppercase; letter-spacing:.05em; }
  .stats div b { font-size:22px; }
  .card { background:var(--panel); border:1px solid var(--line); border-radius:10px; padding:10px 12px; margin-bottom:10px; }
  .card h5 { margin:0 0 6px; font-size:11.5px; letter-spacing:.06em; text-transform:uppercase; color:var(--muted); font-weight:600; }
  #bars { cursor:pointer; display:block; }
  .code { font:12.5px/1.6 "JetBrains Mono","Fira Code",Consolas,monospace; }
  .code div { padding:0 6px; border-radius:4px; white-space:pre-wrap; color:#B8C4D0; }
  .code div.on { background:#1F3A44; color:var(--teal); }
  table { width:100%; border-collapse:collapse; font-size:12.5px; }
  th, td { text-align:left; padding:4px; border-bottom:1px solid var(--line); vertical-align:top; }
  th { color:var(--muted); font-weight:600; }
  tr.best td { color:var(--teal); font-weight:700; }
  .muted { color:var(--muted); font-size:12.5px; }
  .key { display:flex; gap:12px; flex-wrap:wrap; font-size:12px; color:var(--muted); margin:6px 0; }
  .key i { display:inline-block; width:10px; height:10px; border-radius:3px; margin-right:5px; vertical-align:-1px; }
  #tip { position:fixed; pointer-events:none; background:#000C; color:#fff; padding:5px 8px; border-radius:6px; font-size:12px; display:none; }
</style></head>
<body>
<div class="wrap">
  <div>
    <div class="stats">
      <div><span>Sweep k</span><b id="k">0</b></div>
      <div><span>Largest change Δ</span><b id="d">–</b></div>
      <div><span>Greedy moves changed</span><b id="ch">–</b></div>
      <div><span>States</span><b id="ns">–</b></div>
    </div>
    <div class="row">
      <button id="first" aria-label="First sweep">⏮</button>
      <button id="back" aria-label="Previous sweep">◀</button>
      <button id="play" class="go">▶ Play</button>
      <button id="fwd" aria-label="Next sweep">▶</button>
      <button id="last" aria-label="Last sweep">⏭</button>
      <input id="scrub" type="range" min="0" max="0" value="0" aria-label="Sweep">
      <select id="rate" aria-label="Speed"><option value="3">Slow</option><option value="8" selected>Normal</option><option value="20">Fast</option></select>
    </div>
    <div class="key" id="key"></div>
    <canvas id="bars"></canvas>
    <div class="muted" style="margin-top:6px">Each bar is one state, sorted by its final value V*. The bar is V<sub>k</sub>, the white tick is where it ends up, the colour is the best move at this sweep. Click a bar for its Bellman update.</div>
    <div class="card" style="margin-top:10px"><h5>Δ per sweep (log scale)</h5><canvas id="conv"></canvas></div>
  </div>
  <div>
    <div class="card"><h5>Algorithm</h5><div class="code" id="code"></div></div>
    <div class="card"><h5 id="bt">Bellman update</h5><div id="bb"></div></div>
  </div>
</div>
<div id="tip"></div>
<script>
(function () {
"use strict";
const D = /*__DATA__*/null;
const el = id => document.getElementById(id);
const S = D.states.length, A = D.moves.length, F = D.frames;
let fi = 0, playing = false, pick = D.order[0], pulse = 0, rects = [];
el("scrub").max = F.length - 1; el("ns").textContent = S;
el("key").innerHTML = D.moves.map(m => "<span><i style='background:" + D.move_colors[m] + "'></i>" + m + "</span>").join("");

function qRow(s, V) {
  const q = new Array(A).fill(null);
  for (let a = 0; a < A; a++) { if (!D.seen[s][a]) continue; let e = 0; for (const [t, p] of D.P[s][a]) e += p * V[t]; q[a] = D.R[s][a] + D.gamma * e; }
  return q;
}
function greedy(V) { const pi = []; for (let s = 0; s < S; s++) { const q = qRow(s, V); let b = -1, bv = -Infinity; q.forEach((v, a) => { if (v !== null && v > bv) { bv = v; b = a; } }); pi.push(b); } return pi; }
const PI = F.map(f => greedy(f.V));

const cv = el("bars"), g = cv.getContext("2d");
function drawBars() {
  const w = Math.max(300, cv.parentElement.clientWidth), dpr = window.devicePixelRatio || 1;
  const bh = Math.max(4, Math.min(20, Math.floor(620 / S))), gap = bh > 8 ? 3 : 1;
  g.font = "11px system-ui";
  const widest = D.states.reduce((m, s) => Math.max(m, g.measureText(s).width), 0);
  const label = bh >= 12 ? Math.min(Math.ceil(widest) + 10, Math.floor(w * 0.45)) : 8;
  const h = S * (bh + gap) + 24;
  cv.width = w * dpr; cv.height = h * dpr; cv.style.width = w + "px"; cv.style.height = h + "px"; g.setTransform(dpr, 0, 0, dpr, 0, 0);
  g.clearRect(0, 0, w, h);
  const x0 = label + 6, x1 = w - 50, lo = D.vmin, hi = D.vmax;
  const X = v => x0 + (x1 - x0) * (v - lo) / (hi - lo || 1), zero = X(0);
  g.strokeStyle = "#243241"; g.lineWidth = 1; g.beginPath(); g.moveTo(zero, 0); g.lineTo(zero, h - 20); g.stroke();
  g.fillStyle = "#8FA1B3"; g.font = "10px system-ui"; g.textAlign = "center";
  [lo, 0, hi].forEach(v => g.fillText(v.toFixed(1), X(v), h - 6));
  const V = F[fi].V, pi = PI[fi], prev = fi ? PI[fi - 1] : null;
  rects = [];
  D.order.forEach((s, row) => {
    const y = row * (bh + gap);
    if (s === pick) { g.fillStyle = "rgba(46,196,182,0.12)"; g.fillRect(0, y - 1, w, bh + 2); }
    if (label > 8) { g.fillStyle = s === pick ? "#2EC4B6" : "#B8C4D0"; g.font = "11px system-ui"; g.textAlign = "right"; g.textBaseline = "middle"; g.fillText(D.states[s], label, y + bh / 2); }
    const xa = Math.min(zero, X(V[s])), xb = Math.max(zero, X(V[s]));
    g.fillStyle = pi[s] >= 0 ? D.move_colors[D.moves[pi[s]]] : "#555"; g.globalAlpha = 0.9; g.fillRect(xa, y, Math.max(1, xb - xa), bh); g.globalAlpha = 1;
    if (prev && prev[s] !== pi[s]) { g.strokeStyle = "#FFFFFF"; g.lineWidth = 1.5; g.strokeRect(xa + 0.5, y + 0.5, Math.max(1, xb - xa) - 1, bh - 1); }
    g.fillStyle = "#FFFFFF"; g.fillRect(X(D.final[s]) - 1, y - 1, 2, bh + 2);
    rects.push([y, y + bh + gap, s]);
  });
}
const cc = el("conv"), c2 = cc.getContext("2d");
function drawConv() {
  const w = Math.max(280, cc.parentElement.clientWidth - 24), h = 130, dpr = window.devicePixelRatio || 1;
  cc.width = w * dpr; cc.height = h * dpr; cc.style.width = w + "px"; cc.style.height = h + "px"; c2.setTransform(dpr, 0, 0, dpr, 0, 0); c2.clearRect(0, 0, w, h);
  if (!D.conv.length) return;
  const L = 44, Rm = 8, T = 6, B = 18, n = Math.max(2, D.sweeps);
  const logs = D.conv.map(p => Math.log10(Math.max(p[1], 1e-14))).concat([Math.log10(D.tol)]);
  const ymin = Math.min.apply(null, logs), ymax = Math.max.apply(null, logs);
  const X = k => L + (w - L - Rm) * (k - 1) / (n - 1), Y = v => T + (h - T - B) * (ymax - Math.log10(Math.max(v, 1e-14))) / (ymax - ymin || 1);
  c2.fillStyle = "#8FA1B3"; c2.font = "10px system-ui"; c2.textAlign = "right"; c2.textBaseline = "middle";
  for (let e = Math.ceil(ymin); e <= Math.floor(ymax); e += Math.max(1, Math.round((ymax - ymin) / 5))) { const y = Y(Math.pow(10, e)); c2.fillText("1e" + e, L - 4, y); c2.strokeStyle = "#1C2A37"; c2.beginPath(); c2.moveTo(L, y); c2.lineTo(w - Rm, y); c2.stroke(); }
  c2.setLineDash([4, 4]); c2.strokeStyle = "#FF6B6B"; c2.beginPath(); c2.moveTo(L, Y(D.tol)); c2.lineTo(w - Rm, Y(D.tol)); c2.stroke(); c2.setLineDash([]);
  c2.strokeStyle = "#2EC4B6"; c2.lineWidth = 2; c2.beginPath(); D.conv.forEach((p, i) => { const x = X(p[0]), y = Y(p[1]); if (i) c2.lineTo(x, y); else c2.moveTo(x, y); }); c2.stroke();
  const f = F[fi]; if (f.k > 0) { c2.fillStyle = "#FFB347"; c2.beginPath(); c2.arc(X(f.k), Y(f.d), 5, 0, 7); c2.fill(); }
  c2.textAlign = "center"; c2.fillStyle = "#8FA1B3"; c2.fillText("1", L, h - 6); c2.fillText(String(D.sweeps), w - Rm, h - 6);
}
const CODE = ["V(s) = 0 for every state", "repeat (sweep k = 1, 2, ...):", "  for each state s:", "    for each action a seen in s:",
              "      Q[a] = R[s][a] + γ · Σ P[s][a][s'] · V[s']", "    V_new[s] = max(Q)", "  Δ = max |V_new − V|;  V = V_new",
              "until Δ < θ    (θ = " + D.tol.toExponential(0) + ", γ = " + D.gamma + ")", "π*(s) = argmax Q"];
function drawCode() {
  const f = F[fi], last = fi === F.length - 1;
  const on = f.k === 0 ? [0] : (last && D.converged ? [7, 8] : (playing ? [[2, 3, 4], [5], [6]][pulse % 3] : [2, 3, 4, 5, 6]));
  el("code").innerHTML = CODE.map((l, i) => "<div class='" + (on.includes(i) ? "on" : "") + "'>" + l.replace(/</g, "&lt;") + "</div>").join("");
}
const fx = (v, d) => v === null || !isFinite(v) ? "–" : Number(v).toFixed(d);
function drawBellman() {
  const f = F[fi], V = f.V, q = qRow(pick, V);
  let b = -1, bv = -Infinity; q.forEach((v, a) => { if (v !== null && v > bv) { bv = v; b = a; } });
  el("bt").textContent = "Bellman update for " + D.states[pick];
  let h = "<div class='muted'>Uses V<sub>" + f.k + "</sub> to produce V<sub>" + (f.k + 1) + "</sub>. Now V<sub>" + f.k + "</sub>(s) = " + fx(V[pick], 3) + ", visited " + D.visits[pick] + " times in the log.</div>";
  h += "<table><tr><th>Move</th><th>R</th><th>γ · Σ P·V (top 3)</th><th>Q</th></tr>";
  D.moves.forEach((m, a) => {
    if (!D.seen[pick][a]) { h += "<tr><td style='color:#5D6D7E'>" + D.short_moves[m] + "</td><td colspan=3 class='muted'>never taken here in the log</td></tr>"; return; }
    const top = D.P[pick][a].slice().sort((x, y) => y[1] - x[1]);
    const txt = top.slice(0, 3).map(([t, p]) => p.toFixed(2) + "×" + V[t].toFixed(1)).join(" + ") + (top.length > 3 ? " + …" : "");
    h += "<tr class='" + (a === b ? "best" : "") + "'><td><span style='color:" + D.move_colors[m] + "'>■</span> " + D.short_moves[m] + (a === b ? " ★" : "") + "</td><td>" + fx(D.R[pick][a], 2) + "</td><td>" + txt + "</td><td>" + fx(q[a], 2) + "</td></tr>";
  });
  h += "</table><div style='margin-top:6px'>V<sub>" + (f.k + 1) + "</sub>(s) = <b>" + fx(bv, 3) + "</b>" + (b >= 0 ? ", best move <b style='color:" + D.move_colors[D.moves[b]] + "'>" + D.moves[b] + "</b>" : "") + ". Final V* = " + fx(D.final[pick], 3) + "</div>";
  el("bb").innerHTML = h;
}
function render() {
  const f = F[fi];
  el("k").textContent = f.k; el("d").textContent = f.k ? f.d.toExponential(2) : "–";
  el("ch").textContent = fi ? String(PI[fi].reduce((n, a, s) => n + (a !== PI[fi - 1][s] ? 1 : 0), 0)) : "–";
  el("scrub").value = fi; drawBars(); drawConv(); drawCode(); drawBellman();
}
function setPlay(v) { playing = v; el("play").textContent = v ? "⏸ Pause" : "▶ Play"; }
el("play").onclick = () => { if (!playing && fi === F.length - 1) fi = 0; setPlay(!playing); render(); };
el("first").onclick = () => { setPlay(false); fi = 0; render(); };
el("last").onclick = () => { setPlay(false); fi = F.length - 1; render(); };
el("back").onclick = () => { setPlay(false); fi = Math.max(0, fi - 1); render(); };
el("fwd").onclick = () => { setPlay(false); fi = Math.min(F.length - 1, fi + 1); render(); };
el("scrub").oninput = e => { setPlay(false); fi = +e.target.value; render(); };
function rowAt(e) { const r = cv.getBoundingClientRect(), y = e.clientY - r.top; const hit = rects.find(q => y >= q[0] && y < q[1]); return hit ? hit[2] : null; }
cv.addEventListener("click", e => { const s = rowAt(e); if (s !== null) { pick = s; render(); } });
cv.addEventListener("mousemove", e => {
  const s = rowAt(e), tip = el("tip"); if (s === null) { tip.style.display = "none"; return; }
  const a = PI[fi][s]; tip.innerHTML = D.states[s] + "<br>V<sub>" + F[fi].k + "</sub> = " + F[fi].V[s].toFixed(3) + (a >= 0 ? "<br>best: " + D.moves[a] : "");
  tip.style.left = e.clientX + 12 + "px"; tip.style.top = e.clientY + 12 + "px"; tip.style.display = "block";
});
cv.addEventListener("mouseleave", () => el("tip").style.display = "none");
let acc = 0, last = null;
function tick(t) {
  if (last === null) last = t; const dt = Math.min(0.1, (t - last) / 1000); last = t;
  if (playing) { acc += dt * +el("rate").value; if (acc >= 1) { acc = 0; pulse++; if (fi < F.length - 1) fi++; else setPlay(false); render(); } }
  requestAnimationFrame(tick);
}
window.addEventListener("resize", render);
render(); requestAnimationFrame(tick);
})();
</script>
</body></html>


## 5. Theme, requirements and the Streamlit app

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "dark"
primaryColor = "#2EC4B6"
backgroundColor = "#0F1720"
secondaryBackgroundColor = "#16212C"
textColor = "#E6EDF3"
font = "sans serif"

[browser]
gatherUsageStats = false


In [ ]:
%%writefile requirements.txt
streamlit>=1.32
plotly>=5.18
pandas>=2.0
numpy>=1.24


In [ ]:
%%writefile app.py
"""
WallBot Lab - wall-following robot navigation with MDPs
=======================================================
Streamlit front end. The maths lives in engine.py and programs.py, the two
interactive browser components in web/.

Run:  streamlit run app.py
"""

import io
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
import streamlit.components.v1 as components

import engine as E
import programs as PG
import web_bridge as WB

HERE = Path(__file__).parent
CSV_OUT = HERE / "optimal_value_function.csv"

st.set_page_config(page_title="WallBot Lab", page_icon="🛰️", layout="wide")

# ---------------------------------------------------------------------------
# Version-safe helpers
# ---------------------------------------------------------------------------

try:
    _V = tuple(int(x) for x in st.__version__.split(".")[:2])
except Exception:
    _V = (0, 0)
MODERN = _V >= (1, 50)
memo = getattr(st, "cache_data", None) or st.cache


def chart(fig, height=None):
    if height:
        fig.update_layout(height=height)
    if MODERN:
        st.plotly_chart(fig, width="stretch")
    else:
        st.plotly_chart(fig, use_container_width=True)


def table(df, height=None, index=False):
    kw = {"height": height} if height else {}
    kw.update({"width": "stretch"} if MODERN else {"use_container_width": True})
    if not index and _V >= (1, 23):
        kw["hide_index"] = True
    try:
        st.dataframe(df, **kw)
    except TypeError:
        st.dataframe(df)


def editable(df, key):
    ed = getattr(st, "data_editor", None) or getattr(st, "experimental_data_editor", None)
    if ed is None:
        table(df, index=True)
        return df
    try:
        return ed(df, key=key, **({"width": "stretch"} if MODERN else {"use_container_width": True}))
    except TypeError:
        return ed(df, key=key)


def rerun():
    fn = getattr(st, "rerun", None) or getattr(st, "experimental_rerun", None)
    if fn:
        fn()


def embed(name, data, height):
    components.html(WB.load_html(name, data), height=height, scrolling=True)


def callout(text):
    st.markdown(f'<div class="callout">{text}</div>', unsafe_allow_html=True)


def show_code(code, title="Code from the Day 5 notebook"):
    with st.expander(title):
        st.code(code, language="python")


PAPER = "rgba(0,0,0,0)"
INK = "#E6EDF3"
GRIDC = "#243241"
MCOL = WB.MOVE_COLORS


def styled(fig, height=380, title=None):
    fig.update_layout(template="plotly_dark", paper_bgcolor=PAPER, plot_bgcolor="#101A24", height=height,
                      font=dict(color=INK), margin=dict(t=46 if title else 16, b=16, l=10, r=10),
                      title=dict(text=title, font=dict(size=14)) if title else None)
    fig.update_xaxes(gridcolor=GRIDC, zerolinecolor=GRIDC)
    fig.update_yaxes(gridcolor=GRIDC, zerolinecolor=GRIDC)
    return fig


st.markdown("""
<style>
.block-container {padding-top: 1.6rem; max-width: 1300px;}
.callout {background:#16212C; border:1px solid #243241; border-left:4px solid #2EC4B6; padding:.7rem 1rem;
          border-radius:8px; margin-bottom:1rem; color:#C9D4DE;}
div[data-testid="stMetricValue"] {font-size:1.55rem;}
h1, h2, h3 {letter-spacing:-0.01em;}
</style>""", unsafe_allow_html=True)


# ---------------------------------------------------------------------------
# Cached computations
# ---------------------------------------------------------------------------

@memo(show_spinner=False)
def read_data(blob, path):
    return E.load_readings(io.BytesIO(blob) if blob is not None else path)


@memo(show_spinner="Building the MDP and solving it…")
def solve_robot(df, used, cuts_items, gamma, tol, reward_items, smoothing, in_place):
    cuts = {k: list(v) for k, v in cuts_items}
    rs = E.RewardSpec(**dict(reward_items))
    mdp, sol, tab = E.solve(df, list(used), cuts, gamma, tol, rs, smoothing, in_place)
    V_pi, Q_pi, pi_pi, rounds = E.policy_iteration(mdp.P, mdp.R, gamma, mdp.seen)
    return mdp, sol, tab, {"V": V_pi, "pi": pi_pi, "rounds": rounds}


@memo(show_spinner="Replaying the log with Q-learning…")
def run_q_learning(_mdp0, key, gamma, mode, alpha, epochs, seed, reward_items):
    rs = E.RewardSpec(**dict(reward_items))
    _, Q_star, _ = E.fast_value_iteration(_mdp0.P, _mdp0.R, gamma, _mdp0.seen)
    tr = PG.log_transitions(_mdp0, rs)
    Q, hist = PG.offline_q_learning(tr, len(_mdp0.states), len(E.MOVES), _mdp0.seen, gamma, mode, alpha, epochs, seed, Q_star)
    return Q, Q_star, hist


@memo(show_spinner="Simulating episodes…")
def run_mc_eval(n, traps, policy_kind, eps, episodes, gamma, seed):
    rng = np.random.default_rng(seed)
    goal = (n - 1, n - 1)

    def policy(s):
        if policy_kind == "Random (notebook)" or rng.random() < eps:
            return int(rng.integers(4))
        options = ([1] if s[0] < goal[0] else []) + ([3] if s[1] < goal[1] else [])
        return int(rng.choice(options)) if options else int(rng.integers(4))

    env = PG.GridEnvironment((n, n), traps=traps)
    V, cnt, trace, rets, lens, ends = PG.first_visit_mc(env, policy, episodes, gamma)
    sample = [PG.play_episode(PG.GridEnvironment((n, n), traps=traps), policy) for _ in range(12)]
    return V, cnt, trace, rets, lens, ends, sample


@memo(show_spinner="Learning with Monte Carlo control…")
def run_mc_control(n, traps, episodes, gamma, eps, seed):
    env = PG.GridEnvironment((n, n), traps=traps)
    Q, greedy, curve = PG.mc_control(env, episodes, gamma, eps, seed)
    path, end = PG.greedy_rollout(PG.GridEnvironment((n, n), traps=traps), greedy)
    return Q, greedy, curve, path, end


@memo(show_spinner="Mapping the tuning landscape…")
def tuning_landscape(df, used, cuts_items, reward_items, smoothing, n=13):
    tuner = PG.RobotTuner(df, list(used), {k: list(v) for k, v in cuts_items}, E.RewardSpec(**dict(reward_items)), smoothing)
    gs = np.linspace(0.5, 0.99, n)
    ts = np.linspace(0.3, 1.5, n)
    Z = np.array([[tuner.agreement(g, t) for g in gs] for t in ts])
    return gs, ts, Z


# ---------------------------------------------------------------------------
# Sidebar: navigation + robot MDP settings
# ---------------------------------------------------------------------------

PAGES = ["🏠 Overview", "🤖 Simulator", "🎞️ Value iteration player", "📐 MDP (notebook)", "🧠 ADP",
         "🎲 Monte Carlo", "🧭 Hooke-Jeeves", "📊 Data & results"]

with st.sidebar:
    st.markdown("## 🛰️ WallBot Lab")
    page = st.radio("Go to", PAGES, key="page", label_visibility="collapsed")
    st.divider()
    st.markdown("**Robot MDP settings**")
    upload = st.file_uploader("Sensor file (optional)", type=["csv", "data"])
    default_path = E.locate_dataset(__file__)
    if upload is None and default_path is None:
        st.error("sensor_readings_24.csv not found. Put it next to app.py (or in data/), or upload it here.")
        st.stop()
    try:
        df = read_data(upload.getvalue() if upload else None, str(default_path) if default_path else None)
    except Exception as exc:
        st.error(f"Could not read the sensor file: {exc}")
        st.stop()
    if len(df) < 20:
        st.error("The sensor file has too few valid rows.")
        st.stop()

    used = st.multiselect("Distances in the state", E.DIST, default=E.DIST, key="used")
    if not used:
        st.warning("Choose at least one distance.")
        st.stop()
    used = [d for d in E.DIST if d in used]
    level_mode = st.radio("How distances become levels", ["Fixed thresholds (metres)", "Quantiles"], key="level_mode")
    cuts = {}
    if level_mode.startswith("Fixed"):
        with st.expander("Thresholds (Close | Medium | Far)", expanded=False):
            for d in used:
                lo, hi = E.DEFAULT_CUTS[d]
                a, b = st.slider(d, 0.2, 3.0, (float(lo), float(hi)), 0.05, key=f"cut_{d}")
                cuts[d] = [a, b] if b > a else [a, a + 0.05]
    else:
        k = st.select_slider("Levels per distance", [2, 3, 4, 5], value=3, key="q_levels")
        cuts = E.quantile_cuts(df, used, k)
        cuts = {d: (c if len(c) >= 1 else [float(df[d].median())]) for d, c in cuts.items()}

    gamma = st.slider("Discount factor γ", 0.50, 0.99, 0.90, 0.01, key="gamma")
    tol = float(st.select_slider("Convergence threshold θ", ["1e-03", "1e-04", "1e-05", "1e-06", "1e-07", "1e-08"],
                                 value="1e-06", key="tol"))
    smoothing = st.slider("Laplace smoothing α", 0.0, 2.0, 0.5, 0.1, key="smoothing",
                          help="Added to every next state already reached from s, so rare transitions aren't zero.")
    in_place = st.toggle("In-place (Gauss-Seidel) sweeps", value=False, key="in_place") if hasattr(st, "toggle") \
        else st.checkbox("In-place (Gauss-Seidel) sweeps", value=False, key="in_place")
    with st.expander("Reward shaping"):
        d0 = E.RewardSpec()
        rs_vals = {
            "target_left": st.slider("Target left-wall distance (m)", 0.3, 1.5, d0.target_left, 0.05, key="target_left"),
            "tolerance": st.slider("Tolerance around the target (m)", 0.05, 0.6, d0.tolerance, 0.05, key="rw_tol"),
            "follow_gain": st.slider("Wall-following bonus", 0.0, 3.0, d0.follow_gain, 0.1, key="rw_follow"),
            "safe_front": st.slider("Front safety distance (m)", 0.3, 1.5, d0.safe_front, 0.05, key="rw_front"),
            "safe_left": st.slider("Left safety distance (m)", 0.2, 1.0, d0.safe_left, 0.05, key="rw_left"),
            "crash_gain": st.slider("Proximity penalty", 0.0, 10.0, d0.crash_gain, 0.5, key="rw_crash"),
            "forward_bonus": st.slider("Forward bonus", 0.0, 1.0, d0.forward_bonus, 0.05, key="rw_fwd"),
            "sharp_cost": st.slider("Sharp-turn cost", 0.0, 1.0, d0.sharp_cost, 0.05, key="rw_sharp"),
        }

reward = E.RewardSpec(**rs_vals)
cuts_items = tuple((d, tuple(float(x) for x in cuts[d])) for d in used)
reward_items = tuple(rs_vals.items())
mdp, sol, vtab, pi_res = solve_robot(df, tuple(used), cuts_items, gamma, tol, reward_items, smoothing, in_place)
try:
    vtab.to_csv(CSV_OUT, index=False)
    csv_saved = True
except Exception:
    csv_saved = False
dfs = mdp.extras["data"]
best_map = dict(zip(vtab["State"], vtab["Optimal_Action"]))
agree = float((dfs["State"].map(best_map) == dfs["Class"]).mean())
robot_key = (tuple(used), cuts_items, gamma, tol, reward_items, smoothing)


def headline():
    c = st.columns(4)
    c[0].metric("Log readings", f"{len(df):,}")
    c[1].metric("States", len(mdp.states), f"of {int(np.prod([len(cuts[d]) + 1 for d in used]))} possible", delta_color="off")
    c[2].metric("Value-iteration sweeps", sol["sweeps"], "converged" if sol["converged"] else "not converged",
                delta_color="normal" if sol["converged"] else "inverse")
    c[3].metric("Matches the logged move", f"{agree:.0%}")


# ---------------------------------------------------------------------------
# Pages
# ---------------------------------------------------------------------------

def page_overview():
    st.title("WallBot Lab")
    st.caption("Wall-following robot navigation as a Markov Decision Process, solved with value iteration.")
    headline()
    left, right = st.columns([1.25, 1])
    with left:
        st.subheader("What the exercise asks for, and where it lives")
        st.markdown(f"""
| Slide item | In WallBot Lab |
|---|---|
| **States** | {", ".join(used)} turned into levels ({level_mode.lower()}); {len(mdp.states)} combinations occur in the log |
| **Action space** | {", ".join(E.MOVES)} |
| **Transition probabilities** | Counted from consecutive readings (9 per second) with Laplace smoothing α = {smoothing} |
| **Rewards** | Gaussian bonus around {reward.target_left:.2f} m from the left wall, graded proximity penalties, forward bonus, sharp-turn cost |
| **Discount factor γ** | {gamma} |
| **Convergence threshold** | {tol:.0e}, value iteration stops when no value changes by more than this |
| **Output** | `optimal_value_function.csv` (download on the Data & results page) |
""")
        callout("Use the sidebar to change any MDP setting. Every page updates straight away: the simulator's robot, the "
                "value iteration player, the results table and the CSV.")
    with right:
        st.subheader("Pipeline")
        try:
            st.graphviz_chart("""digraph { rankdir=TB; bgcolor="transparent";
              node [shape=box, style="rounded,filled", fillcolor="#16212C", color="#2EC4B6", fontcolor="#E6EDF3", fontname="Helvetica", fontsize=11];
              edge [color="#8FA1B3"];
              a [label="24 ultrasound sensors"]; b [label="4 simplified distances (min of each arc)"];
              c [label="Levels -> MDP states"]; d [label="P(s'|s,a) with smoothing, R(s,a) shaped"];
              e [label="Value iteration (θ, γ)"]; f [label="V*, π*  ->  optimal_value_function.csv"];
              a -> b -> c -> d -> e -> f; }""")
        except Exception:
            st.write("24 sensors → 4 distances → states → P and R → value iteration → V*, π*")
        counts = df["Class"].value_counts().reindex(E.MOVES).fillna(0)
        fig = go.Figure(go.Pie(labels=counts.index, values=counts.values, hole=0.55,
                               marker=dict(colors=[MCOL[m] for m in counts.index])))
        chart(styled(fig, 300, "Moves recorded in the log"))


def page_simulator():
    st.title("🤖 Simulator")
    callout("Place a <b>start</b> (●) and a <b>finish</b> (⚑), then press Run. The <b>learned wall-follower</b> reads its "
            "four sensor arcs, turns them into an MDP state and takes the optimal move from value iteration. The "
            "<b>goal seeker</b> plans on the grid itself: its own MDP with 4 moves and slip, solved by value iteration "
            "in your browser. The chart under the room tracks the left and front distances against the target band.")
    embed("simulator.html", WB.simulator_data(mdp, sol, reward), 1000)
    with st.expander("How the simulator works"):
        st.markdown(f"""
- **Room:** 20 × 14 squares of 0.30 m (6.0 m × 4.2 m). Draw or erase walls by clicking and dragging.
- **Sensing:** 13 rays per 60° arc; the shortest is the simplified distance, just like SD_front, SD_left, SD_right and SD_back in the dataset.
- **State and action:** the distances are cut with the same thresholds as the MDP, and the move comes from the optimal policy π*.
- **Finish:** the run ends when the rover is within 0.35 m of the flag. The wall-follower only passes finishes along the wall it follows (clockwise, wall on its left). In the maze it can't get through, which is where the goal seeker comes in.
- **Goal seeker MDP:** states are free squares, actions north/east/south/west, each move costs 1, bumping a wall costs 3 more, γ = 0.98. With slip p the rover goes sideways with probability p/2 each way, and the plan accounts for that.
- **Rewards shown** use the same shaping as the sidebar (target {reward.target_left:.2f} m).
""")


def page_vi():
    st.title("🎞️ Value iteration player")
    callout("Press Play to watch every state's value grow sweep by sweep, and the best move settle. Click a bar to see "
            "its Bellman update worked out with real numbers.")
    embed("vi_player.html", WB.vi_player_data(mdp, sol, gamma, tol), 1080)
    c = st.columns(3)
    other = E.value_iteration(mdp.P, mdp.R, gamma, tol, seen=mdp.seen, in_place=not in_place)
    c[0].metric("Sweeps (current setting)", sol["sweeps"], "in-place" if in_place else "standard", delta_color="off")
    c[1].metric("Sweeps with the other update", other["sweeps"], "standard" if in_place else "in-place", delta_color="off")
    c[2].metric("Policy iteration rounds", len(pi_res["rounds"]),
                f"same policy: {np.mean(pi_res['pi'] == sol['pi']):.0%}", delta_color="off")
    show_code('''def value_iteration(P, R, gamma, tolerance=1e-6, max_iterations=20_000, seen=None, in_place=False):
    S, A = R.shape
    V = np.zeros(S)
    for _ in range(max_iterations):
        V_old = V.copy()
        V_new = V if in_place else np.zeros(S)
        for s in range(S):
            Q_sa = np.full(A, -np.inf)
            for a in range(A):
                if seen[s, a]:                                   # only moves recorded in the log
                    Q_sa[a] = R[s][a] + gamma * np.dot(P[s][a], V)
            V_new[s] = np.max(Q_sa)
        V = V_new
        if np.max(np.abs(V - V_old)) < tolerance:               # convergence threshold
            break
    return V''', "Value iteration used here (same update as the notebook)")


def page_mdp():
    st.title("📐 MDP: the notebook example")
    callout("The Day 5 notebook's three-state, two-action MDP. Edit any number and it is solved again with "
            "<b>value iteration</b> and <b>policy iteration</b>, which should always agree.")
    left, right = st.columns([1, 1.15])
    with left:
        st.markdown("**Rewards R(s, a)**")
        r_df = editable(pd.DataFrame(PG.NB_R, index=PG.NB_STATES, columns=PG.NB_ACTIONS), "nb_R")
        st.markdown("**Transitions P(s' | s, a)**, each row is rescaled to sum to 1")
        rows = [f"{s} · {a}" for s in PG.NB_STATES for a in PG.NB_ACTIONS]
        t_df = editable(pd.DataFrame(PG.NB_T.reshape(6, 3), index=rows, columns=PG.NB_STATES), "nb_T")
        g_nb = st.slider("γ", 0.0, 0.99, 0.9, 0.01, key="nb_gamma")
        mode = st.radio("Stop after", ["1000 sweeps (notebook)", "Δ below 1e-6"], horizontal=True, key="nb_mode")
    try:
        R = pd.DataFrame(r_df).apply(pd.to_numeric, errors="coerce").fillna(0).to_numpy(float).reshape(3, 2)
        T, fixed = PG.normalise_rows(pd.DataFrame(t_df).apply(pd.to_numeric, errors="coerce").fillna(0).to_numpy(float).reshape(3, 2, 3))
    except Exception:
        st.warning("The tables could not be read; the notebook values are used instead.")
        R, (T, fixed) = PG.NB_R.copy(), (PG.NB_T.copy(), [])
    V, Q, trace, deltas = PG.notebook_value_iteration(T, R, g_nb, 1000, None if mode.startswith("1000") else 1e-6)
    rounds = PG.notebook_policy_iteration(T, R, g_nb)
    pi = Q.argmax(1)
    with right:
        if fixed:
            st.caption("Rescaled rows: " + ", ".join(f"{PG.NB_STATES[s]} · {PG.NB_ACTIONS[a]}" for s, a in fixed))
        m = st.columns(3)
        for i, s in enumerate(PG.NB_STATES):
            m[i].metric(f"V*({s})", f"{V[i]:.2f}", f"take {PG.NB_ACTIONS[pi[i]]}", delta_color="off")
        # Sankey of the optimal policy's transitions
        src, tgt, val, colr, lab = [], [], [], [], []
        for s in range(3):
            for t in range(3):
                p = T[s, pi[s], t]
                if p > 0.001:
                    src.append(s); tgt.append(3 + t); val.append(p)
                    colr.append("rgba(46,196,182,0.45)" if pi[s] == 0 else "rgba(255,179,71,0.45)")
                    lab.append(f"{PG.NB_STATES[s]} --{PG.NB_ACTIONS[pi[s]]}--> {PG.NB_STATES[t]}: {p:.2f}")
        fig = go.Figure(go.Sankey(
            node=dict(label=[f"{s} (now)" for s in PG.NB_STATES] + [f"{s} (next)" for s in PG.NB_STATES],
                      color=["#2EC4B6", "#5AA9E6", "#9D8DF1"] * 2, pad=18, thickness=16),
            link=dict(source=src, target=tgt, value=val, color=colr, label=lab)))
        chart(styled(fig, 330, "Where the optimal policy leads (teal = a1, amber = a2)"))
        st.caption(f"Value iteration stopped after {len(deltas)} sweeps. Policy iteration needed {len(rounds)} round(s) "
                   f"and found the same policy: {'yes' if (rounds[-1]['policy'] == pi).all() else 'no'}.")
    st.subheader("Convergence")
    a, b = st.columns([1.3, 1])
    with a:
        kmax = min(len(trace) - 1, 120)
        fig = go.Figure()
        for i, s in enumerate(PG.NB_STATES):
            fig.add_trace(go.Scatter(x=np.arange(kmax + 1), y=trace[: kmax + 1, i], name=s, mode="lines",
                                     line=dict(width=2.5, color=["#2EC4B6", "#5AA9E6", "#9D8DF1"][i])))
        k = st.slider("Inspect sweep k", 0, kmax, min(3, kmax), key="nb_k")
        fig.add_vline(x=k, line_dash="dot", line_color="#FFB347")
        chart(styled(fig, 320, "V_k(s) for the first sweeps"))
    with b:
        Vk = trace[k]
        rows = []
        for s in range(3):
            q = [R[s, a] + g_nb * float(T[s, a] @ Vk) for a in range(2)]
            for a in range(2):
                rows.append({"s": PG.NB_STATES[s], "a": PG.NB_ACTIONS[a],
                             "R + γ·Σ P·V_k": f"{R[s, a]:.1f} + {g_nb:.2f}·({' + '.join(f'{T[s, a, j]:.2f}×{Vk[j]:.1f}' for j in range(3))})",
                             "Q": round(q[a], 3), "best": "★" if a == int(np.argmax(q)) else ""})
        st.markdown(f"**Bellman update at sweep {k}**")
        table(pd.DataFrame(rows))
        pi_rows = [{"Round": r["round"], "Policy": ", ".join(f"{PG.NB_STATES[i]}→{PG.NB_ACTIONS[a]}" for i, a in enumerate(r["policy"])),
                    "V": ", ".join(f"{v:.2f}" for v in r["V"])} for r in rounds]
        st.markdown("**Policy iteration, round by round**")
        table(pd.DataFrame(pi_rows))
    show_code('''states = ['s1', 's2', 's3']; actions = ['a1', 'a2']
R = np.array([[5, 10], [2, 3], [8, 1]])
T = np.array([[[0.7, 0.2, 0.1], [0.1, 0.6, 0.3]],
              [[0.3, 0.4, 0.3], [0.5, 0.3, 0.2]],
              [[0.4, 0.4, 0.2], [0.2, 0.5, 0.3]]])
gamma = 0.9

def value_iteration(T, R, gamma, iterations=1000):
    V = np.zeros(len(states))
    for i in range(iterations):
        V_new = np.zeros(len(states))
        for s in range(len(states)):
            Q_sa = np.zeros(len(actions))
            for a in range(len(actions)):
                Q_sa[a] = R[s][a] + gamma * np.dot(T[s][a], V)
            V_new[s] = np.max(Q_sa)
        V = V_new
    return V''')


def page_adp():
    st.title("🧠 Approximate Dynamic Programming")
    callout("Exact value iteration needs the full model P and one value per state. ADP relaxes one of those. "
            "<b>Offline Q-learning</b> learns straight from the logged (s, a, r, s') samples with no model at all. "
            "<b>State aggregation</b> solves a smaller MDP by merging states, then uses those values for the full one.")
    method = st.radio("Method", ["Offline Q-learning from the robot log", "State aggregation"], horizontal=True, key="adp_m")
    if method.startswith("Offline"):
        c = st.columns(4)
        epochs = c[0].slider("Passes over the log (epochs)", 5, 60, 25, 5, key="ql_ep")
        rate = c[1].radio("Learning rate", ["Decaying 1/n(s,a)^0.6", "Constant α"], key="ql_mode")
        amode = "decaying" if rate.startswith("Decaying") else "constant"
        alpha = c[2].slider("Constant α", 0.01, 0.5, 0.05, 0.01, key="ql_alpha", disabled=amode == "decaying")
        seed = c[3].number_input("Shuffle seed", 0, 999, 0, key="ql_seed")
        mdp0 = E.build_robot_mdp(df, used, cuts, reward, smoothing=0.0)
        Q, Q_star, h = run_q_learning(mdp0, robot_key, gamma, amode, alpha, epochs, int(seed), reward_items)
        seen = mdp0.seen
        m = st.columns(4)
        m[0].metric("Samples replayed per epoch", f"{len(df) - 1:,}")
        m[1].metric("Max |Q − Q*|", f"{h['max_error'][-1]:.3f}")
        m[2].metric("Mean |Q − Q*|", f"{h['mean_error'][-1]:.3f}")
        m[3].metric("Same best move as Q*", f"{h['agreement'][-1]:.0%}")
        a, b = st.columns(2)
        with a:
            fig = go.Figure([go.Scatter(x=h["epoch"], y=h["max_error"], name="max error", line=dict(color="#FF6B6B", width=2.5)),
                             go.Scatter(x=h["epoch"], y=h["mean_error"], name="mean error", line=dict(color="#2EC4B6", width=2.5))])
            fig.update_yaxes(type="log")
            chart(styled(fig, 340, "Error against the exact Q* after each pass"))
        with b:
            qs, qe = Q_star[seen], Q[seen]
            fig = go.Figure(go.Scatter(x=qs, y=qe, mode="markers", marker=dict(color="#5AA9E6", size=7, opacity=0.8)))
            lo, hi = float(min(qs.min(), qe.min())), float(max(qs.max(), qe.max()))
            fig.add_shape(type="line", x0=lo, y0=lo, x1=hi, y1=hi, line=dict(color="#8FA1B3", dash="dash"))
            fig.update_xaxes(title="exact Q*(s,a)"); fig.update_yaxes(title="learned Q(s,a)")
            chart(styled(fig, 340, "Every (state, move) pair"))
        st.caption("Q* here is the exact solution of the same log without smoothing, the model Q-learning is implicitly "
                   "learning. A decaying learning rate averages out the randomness of individual transitions; a "
                   "constant one keeps chasing it.")
    else:
        keep = st.multiselect("Distances the aggregated MDP keeps", used, default=used[:2], key="agg_keep")
        V_ex, _, pi_ex = E.fast_value_iteration(mdp.P, mdp.R, gamma, mdp.seen)
        res = PG.state_aggregation(mdp, keep, gamma)
        m = st.columns(4)
        m[0].metric("States in the small MDP", res["groups"], f"vs {len(mdp.states)}", delta_color="off")
        m[1].metric("Max |V − V*|", f"{np.abs(res['V'] - V_ex).max():.3f}")
        m[2].metric("Mean |V − V*|", f"{np.abs(res['V'] - V_ex).mean():.3f}")
        m[3].metric("Same best move as exact", f"{np.mean(res['pi'] == pi_ex):.0%}")
        rows = []
        for r in range(len(used) + 1):
            for sub in itertools.combinations(used, r):
                out = PG.state_aggregation(mdp, list(sub), gamma)
                rows.append({"Kept": ", ".join(x.replace("SD_", "") for x in sub) or "(nothing)", "States": out["groups"],
                             "Mean error": float(np.abs(out["V"] - V_ex).mean()), "Policy match": float(np.mean(out["pi"] == pi_ex))})
        tab = pd.DataFrame(rows).sort_values("States")
        a, b = st.columns(2)
        with a:
            fig = go.Figure(go.Scatter(x=tab["States"], y=tab["Mean error"], mode="markers+text", text=tab["Kept"],
                                       textposition="top center", marker=dict(size=11, color=tab["Policy match"],
                                       colorscale=[[0, "#FF6B6B"], [1, "#2EC4B6"]], showscale=True, colorbar=dict(title="match"))))
            fig.update_xaxes(title="states in the aggregated MDP", type="log"); fig.update_yaxes(title="mean |V − V*|")
            chart(styled(fig, 400, "Fewer states, more error: every possible aggregation"))
        with b:
            fig = go.Figure(go.Scatter(x=V_ex, y=res["V"], mode="markers", marker=dict(color="#9D8DF1", size=8)))
            lo, hi = float(min(V_ex.min(), res["V"].min())), float(max(V_ex.max(), res["V"].max()))
            fig.add_shape(type="line", x0=lo, y0=lo, x1=hi, y1=hi, line=dict(color="#8FA1B3", dash="dash"))
            fig.update_xaxes(title="exact V*"); fig.update_yaxes(title="aggregated V")
            chart(styled(fig, 400, "Your selection, state by state"))
        st.caption("The best move is chosen with a one-step look-ahead through the full model, so even rough values "
                   "often pick the right move: most of the decision comes from the immediate reward.")
    show_code('''# "Approximate Dynamic Programming" cell of the Day 5 notebook
def value_iteration(T, R, gamma, iterations=1000):
    V = np.zeros(len(states))
    for i in range(iterations):
        V_new = np.zeros(len(states))
        for s in range(len(states)):
            Q_sa = np.zeros(len(actions))
            for a in range(len(actions)):
                Q_sa[a] = R[s][a] + gamma * np.dot(T[s][a], V)
            V_new[s] = np.max(Q_sa)
        V = V_new
    return V
# Offline Q-learning replaces np.dot(T[s][a], V) by one logged sample r + gamma * max Q(s', .)''')


def grid_animation(n, traps, pts, title):
    base = np.zeros((n, n))
    for t in traps:
        base[t] = -1
    base[n - 1, n - 1] = 1
    xs, ys = [p[1] for p in pts][:300], [p[0] for p in pts][:300]
    bg = go.Heatmap(z=base, showscale=False, hoverinfo="skip", xgap=3, ygap=3, zmin=-1, zmax=1,
                    colorscale=[[0, "#5A2A2E"], [0.5, "#1B2733"], [1, "#1F4D47"]])
    trail = lambda i: go.Scatter(x=xs[: i + 1], y=ys[: i + 1], mode="lines", line=dict(color="#5AA9E6", width=3))
    bot = lambda i: go.Scatter(x=[xs[i]], y=[ys[i]], mode="markers", marker=dict(size=24, color="#FFB347", symbol="diamond", line=dict(color="#0F1720", width=2)))
    fig = go.Figure(data=[bg, trail(0), bot(0)], frames=[go.Frame(data=[bg, trail(i), bot(i)], name=str(i)) for i in range(len(xs))])
    ann = [dict(x=0, y=0, text="S", showarrow=False, font=dict(color="#2EC4B6", size=15)),
           dict(x=n - 1, y=n - 1, text="G", showarrow=False, font=dict(color="#2EC4B6", size=15))]
    ann += [dict(x=t[1], y=t[0], text="✕", showarrow=False, font=dict(color="#FF6B6B", size=15)) for t in traps]
    styled(fig, 500, title)
    fig.update_layout(showlegend=False, annotations=ann, margin=dict(t=70, b=80, l=10, r=10),
                      xaxis=dict(range=[-0.5, n - 0.5], dtick=1, title="Y", constrain="domain", showgrid=False),
                      yaxis=dict(range=[n - 0.5, -0.5], dtick=1, title="X", scaleanchor="x", constrain="domain", showgrid=False),
                      updatemenus=[dict(type="buttons", x=1, y=1.02, xanchor="right", yanchor="bottom", direction="left", showactive=False,
                                        buttons=[dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=170, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                                 dict(label="⏸", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))])])],
                      sliders=[dict(x=0, len=1, y=-0.12, yanchor="top", currentvalue=dict(prefix="Step "),
                                    steps=[dict(label=str(i), method="animate", args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for i in range(len(xs))])])
    return fig


def page_mc():
    st.title("🎲 Monte Carlo")
    callout("The notebook's grid world: start at S, +10 at G, −1 per step, ✕ traps cost −10 and end the episode. "
            "Monte Carlo methods learn from complete episodes: <b>evaluation</b> estimates how good a policy is, "
            "<b>control</b> improves the policy as it goes.")
    c = st.columns(4)
    n = c[0].slider("Grid size", 4, 10, 5, key="mc_n")
    n_traps = c[1].slider("Traps", 0, 12, 3, key="mc_traps")
    layout_seed = c[2].number_input("Trap layout", 0, 999, 4, key="mc_layout")
    gamma_mc = c[3].slider("γ", 0.5, 0.99, 0.9, 0.01, key="mc_gamma")
    traps = PG.place_traps(n, n_traps, int(layout_seed))
    mode = st.radio("Task", ["Evaluate a policy (first-visit Monte Carlo)", "Learn a policy (Monte Carlo control)"], horizontal=True, key="mc_mode")
    if mode.startswith("Evaluate"):
        c = st.columns(3)
        kind = c[0].selectbox("Policy", ["Random (notebook)", "Head for the goal, ε random"], key="mc_kind")
        eps = c[1].slider("ε", 0.0, 1.0, 0.3, 0.05, key="mc_eps", disabled=kind.startswith("Random"))
        episodes = c[2].select_slider("Episodes", [200, 500, 1000, 2000, 5000], value=1000, key="mc_episodes")
        V, cnt, trace, rets, lens, ends, sample = run_mc_eval(n, tuple(traps), kind, eps, episodes, gamma_mc, 11)
        m = st.columns(4)
        m[0].metric("Estimated V(start)", f"{trace[-1]:.2f}")
        m[1].metric("Mean episode length", f"{lens.mean():.1f}")
        m[2].metric("Reached the goal", f"{ends.count('goal') / len(ends):.0%}")
        m[3].metric("Fell in a trap", f"{ends.count('trap') / len(ends):.0%}")
        a, b = st.columns(2)
        with a:
            txt = np.where(np.isnan(V), "", np.vectorize(lambda v: f"{v:.1f}")(np.nan_to_num(V)))
            for t in traps:
                txt[t] = "✕"
            txt[n - 1, n - 1] = "G"
            fig = go.Figure(go.Heatmap(z=V, text=txt, texttemplate="%{text}", colorscale=[[0, "#FF6B6B"], [0.5, "#1B2733"], [1, "#2EC4B6"]],
                                       xgap=3, ygap=3, colorbar=dict(title="V")))
            fig.update_yaxes(autorange="reversed", title="X"); fig.update_xaxes(title="Y")
            chart(styled(fig, 420, "Value of every square under this policy"))
        with b:
            fig = go.Figure(go.Scatter(x=np.arange(1, len(trace) + 1), y=trace, line=dict(color="#2EC4B6", width=2.5)))
            fig.update_xaxes(title="episodes"); fig.update_yaxes(title="estimate of V(start)")
            chart(styled(fig, 200, "The estimate settles as episodes accumulate"))
            fig = go.Figure(go.Histogram(x=rets, nbinsx=40, marker_color="#9D8DF1"))
            fig.update_xaxes(title="return from the start")
            chart(styled(fig, 200, "Returns of all episodes"))
        pick = st.slider("Replay sample episode", 1, len(sample), 1, key="mc_pick")
        steps, final, end = sample[pick - 1]
        pts = [s for s, _, _ in steps] + [final]
        chart(grid_animation(n, traps, pts, f"Episode {pick}: {len(steps)} steps, ended at {end}"))
    else:
        c = st.columns(2)
        eps = c[0].slider("Exploration ε", 0.01, 0.5, 0.2, 0.01, key="mcc_eps")
        episodes = c[1].select_slider("Training episodes", [300, 1000, 2000, 4000], value=2000, key="mcc_eps_n")
        Q, greedy, curve, path, end = run_mc_control(n, tuple(traps), episodes, gamma_mc, eps, 5)
        m = st.columns(3)
        m[0].metric("Success rate, last 50 episodes", f"{curve[-1][1]:.0%}" if curve else "–")
        m[1].metric("Greedy policy result", end)
        m[2].metric("Greedy path length", f"{len(path) - 1} steps")
        a, b = st.columns(2)
        with a:
            vmax = Q.max(axis=2)
            txt = np.vectorize(lambda a: PG.ARROWS[a])(greedy).astype(object)
            for t in traps:
                txt[t] = "✕"
            txt[n - 1, n - 1] = "G"
            fig = go.Figure(go.Heatmap(z=vmax, text=txt, texttemplate="%{text}", textfont=dict(size=18), xgap=3, ygap=3,
                                       colorscale=[[0, "#FF6B6B"], [0.5, "#1B2733"], [1, "#2EC4B6"]], colorbar=dict(title="max Q")))
            fig.update_yaxes(autorange="reversed", title="X"); fig.update_xaxes(title="Y")
            chart(styled(fig, 430, "Learned policy (arrows) and its values"))
        with b:
            if curve:
                fig = go.Figure(go.Scatter(x=[c_[0] for c_ in curve], y=[c_[1] for c_ in curve], line=dict(color="#2EC4B6", width=2.5), fill="tozeroy", fillcolor="rgba(46,196,182,0.15)"))
                fig.update_yaxes(range=[0, 1.05], tickformat=".0%", title="episodes reaching G"); fig.update_xaxes(title="training episodes")
                chart(styled(fig, 430, "Learning curve (50-episode windows)"))
        chart(grid_animation(n, traps, path, f"Following the learned greedy policy: ended at {end}"))
    show_code('''class MonteCarloPolicySearch:
    def __init__(self, env, policy, gamma=0.99):
        self.env = env; self.policy = policy; self.gamma = gamma

    def generate_episode(self):
        episode = []
        state = self.env.reset()
        while True:
            action = self.policy(state)
            next_state, reward, done, _ = self.env.step(action)
            episode.append((state, action, reward))
            if done:
                break
            state = next_state
        return episode

    def evaluate_policy(self, num_episodes=1000):
        returns = []
        for _ in range(num_episodes):
            episode = self.generate_episode()
            G = 0
            for t in reversed(range(len(episode))):
                state, action, reward = episode[t]
                G = reward + self.gamma * G
            returns.append(G)
        return np.mean(returns)''')


def hj_contour(func, box, minima, log, probes, x0, title, zlabel="f", transform=None):
    x_lo, x_hi, y_lo, y_hi = box
    gx, gy = np.linspace(x_lo, x_hi, 150), np.linspace(y_lo, y_hi, 150)
    XX, YY = np.meshgrid(gx, gy)
    Z = func([XX, YY])
    Z = transform(Z) if transform else Z
    path_x = [x0[0]] + [r["x"][0] for r in log]
    path_y = [x0[1]] + [r["x"][1] for r in log]
    cont = go.Contour(x=gx, y=gy, z=Z, colorscale=[[0, "#0E3B43"], [0.5, "#2A5D6B"], [1, "#E8A15B"]], showscale=False,
                      line=dict(width=0.5, color="rgba(230,237,243,0.25)"), hoverinfo="skip")
    mins = go.Scatter(x=[m[0] for m in minima], y=[m[1] for m in minima], mode="markers",
                      marker=dict(symbol="star", size=15, color="#FFFFFF", line=dict(color="#0F1720", width=1)))
    pr = go.Scatter(x=probes[:, 0] if len(probes) else [], y=probes[:, 1] if len(probes) else [], mode="markers",
                    marker=dict(size=4, color="rgba(230,237,243,0.35)"))

    def at(i):
        rec = log[i]
        b, d = rec["base"], rec["step"]
        cross = go.Scatter(x=[b[0] - d, b[0] + d, None, b[0], b[0]], y=[b[1], b[1], None, b[1] - d, b[1] + d],
                           mode="lines", line=dict(color="#FFB347", width=3))
        return [cont, pr, mins, go.Scatter(x=path_x[: i + 2], y=path_y[: i + 2], mode="lines+markers",
                                            line=dict(color="#2EC4B6", width=2), marker=dict(size=6)), cross,
                go.Scatter(x=[rec["x"][0]], y=[rec["x"][1]], mode="markers", marker=dict(size=15, color="#FFB347", line=dict(color="#0F1720", width=2)))]

    keep = list(range(len(log))) if len(log) <= 120 else sorted(set(np.linspace(0, len(log) - 1, 120).astype(int).tolist()))
    frames = [go.Frame(data=at(i), name=str(i), layout=go.Layout(title=dict(text=f"Iteration {i + 1}: {log[i]['move']}, step {log[i]['step']:.3g}"))) for i in keep]
    fig = go.Figure(data=at(len(log) - 1), frames=frames)
    styled(fig, 600, title)
    fig.update_layout(showlegend=False, margin=dict(t=70, b=80, l=10, r=10),
                      xaxis=dict(range=[x_lo, x_hi], constrain="domain", showgrid=False),
                      yaxis=dict(range=[y_lo, y_hi], scaleanchor="x", constrain="domain", showgrid=False),
                      updatemenus=[dict(type="buttons", x=1, y=1.02, xanchor="right", yanchor="bottom", direction="left", showactive=False,
                                        buttons=[dict(label="▶ Replay", method="animate", args=[None, dict(frame=dict(duration=240, redraw=True), fromcurrent=False, transition=dict(duration=0))]),
                                                 dict(label="⏸", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))])])],
                      sliders=[dict(x=0, len=1, y=-0.1, yanchor="top", currentvalue=dict(prefix="Iteration "),
                                    steps=[dict(label=str(i + 1), method="animate", args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for i in keep])])
    return fig


def page_hj():
    st.title("🧭 Hooke-Jeeves pattern search")
    callout("A derivative-free optimiser: probe each coordinate by ± step (the orange cross), jump further in any "
            "direction that helped (the <b>pattern move</b>), and halve the step when nothing helps. Try it on test "
            "functions, or let it <b>tune the robot MDP itself</b>.")
    mode = st.radio("Use it on", ["Test functions", "Tune the robot MDP"], horizontal=True, key="hj_mode")
    if mode == "Test functions":
        c = st.columns(3)
        name = c[0].selectbox("Function", list(PG.TEST_FUNCTIONS), key="hj_fn")
        func, box, minima = PG.TEST_FUNCTIONS[name]
        default = (1.0, 1.0) if "notebook" in name else (box[0] + 0.25 * (box[1] - box[0]), box[2] + 0.8 * (box[3] - box[2]))
        x0 = c[1].slider("Start x", float(box[0]), float(box[1]), float(default[0]), 0.1, key=f"hj_x_{name}")
        y0 = c[2].slider("Start y", float(box[2]), float(box[3]), float(default[1]), 0.1, key=f"hj_y_{name}")
        c = st.columns(4)
        step = c[0].slider("Initial step", 0.05, 2.0, 0.5, 0.05, key="hj_step")
        eps = float(c[1].select_slider("Stop when step <", ["1e-02", "1e-03", "1e-04", "1e-06", "1e-08"], value="1e-06", key="hj_eps"))
        iters = c[2].slider("Max iterations", 10, 3000, 1000, 10, key="hj_iters")
        verify = c[3].checkbox("Check the pattern move first", value=True, key="hj_verify",
                               help="Untick to run exactly like the notebook, which always keeps the pattern move.")
        x, log, probes, calls = PG.hooke_jeeves(func, [x0, y0], step, eps, iters, verify)
        m = st.columns(4)
        m[0].metric("Optimized parameters", f"({x[0]:.4f}, {x[1]:.4f})")
        m[1].metric("f(x)", f"{func(x):.2e}")
        m[2].metric("Iterations / evaluations", f"{len(log)} / {calls}")
        m[3].metric("Distance to nearest minimum", f"{min(np.hypot(x[0] - a, x[1] - b) for a, b in minima):.2e}")
        if not verify and len(log) >= iters and log[-1]["step"] > eps:
            st.warning("The notebook version used every iteration without shrinking its step: the pattern move keeps "
                       "overshooting, so the search bounces back and forth. Tick 'Check the pattern move first'.")
        if not log:
            st.info("The step is already below the stopping size, so there is nothing to do.")
            return
        chart(hj_contour(func, box, minima, log, probes, (x0, y0), "Search path", transform=lambda Z: np.log10(Z - Z.min() + 1)))
        hist = pd.DataFrame([{"iteration": r["iter"], "f": r["f"], "step": r["step"], "move": r["move"]} for r in log])
        a, b = st.columns(2)
        with a:
            fig = go.Figure(go.Scatter(x=hist["iteration"], y=np.maximum(hist["f"] - min(0.0, hist["f"].min()), 1e-16), line=dict(color="#2EC4B6", width=2.5)))
            fig.update_yaxes(type="log")
            chart(styled(fig, 280, "Objective value"))
        with b:
            fig = go.Figure(go.Scatter(x=hist["iteration"], y=hist["step"], line=dict(color="#FFB347", width=2.5), line_shape="hv"))
            fig.add_hline(y=eps, line_dash="dash", line_color="#FF6B6B")
            fig.update_yaxes(type="log")
            chart(styled(fig, 280, "Step size (halves when stuck)"))
        with st.expander("Iteration log"):
            table(hist.round(6), height=300)
    else:
        st.markdown("Hooke-Jeeves searches over the **discount factor γ** and the **target wall distance** to make the "
                    "optimal policy agree as often as possible with what the real robot did. Every evaluation "
                    "rebuilds the rewards and re-runs value iteration.")
        c = st.columns(4)
        g0 = c[0].slider("Start γ", 0.5, 0.99, float(gamma), 0.01, key="tune_g0")
        t0 = c[1].slider("Start target (m)", 0.3, 1.5, float(reward.target_left), 0.05, key="tune_t0")
        step = c[2].slider("Initial step", 0.02, 0.3, 0.1, 0.01, key="tune_step")
        iters = c[3].slider("Max iterations", 5, 80, 40, 5, key="tune_iters")
        tuner = PG.RobotTuner(df, used, cuts, reward, smoothing)
        x, log, probes, calls = PG.hooke_jeeves(lambda v: -tuner.agreement(v[0], v[1]), [g0, t0], step, 0.004, iters, True)
        gb, tb = tuner.clip(*x)
        start_score, best_score = tuner.agreement(g0, t0), tuner.agreement(gb, tb)
        m = st.columns(4)
        m[0].metric("Best γ", f"{gb:.3f}")
        m[1].metric("Best target distance", f"{tb:.2f} m")
        m[2].metric("Agreement with the log", f"{best_score:.1%}", f"{(best_score - start_score) * 100:+.1f} pts vs start")
        m[3].metric("Distinct MDPs solved", tuner.calls)
        gs, ts, Z = tuning_landscape(df, tuple(used), cuts_items, reward_items, smoothing)
        land = go.Contour(x=gs, y=ts, z=Z, colorscale=[[0, "#0E3B43"], [0.5, "#2A5D6B"], [1, "#E8A15B"]],
                          colorbar=dict(title="agreement", tickformat=".0%"), line=dict(width=0.5, color="rgba(230,237,243,0.25)"))
        clipped = [tuner.clip(*r["x"]) for r in log]
        fig = go.Figure([land, go.Scatter(x=[g0] + [p[0] for p in clipped], y=[t0] + [p[1] for p in clipped], mode="lines+markers",
                                          line=dict(color="#FFFFFF", width=2), marker=dict(size=7, color="#FFB347"))])
        fig.update_xaxes(title="discount factor γ"); fig.update_yaxes(title="target left-wall distance (m)")
        chart(styled(fig, 520, "Agreement with the logged moves, and the Hooke-Jeeves path"))

        def apply():
            st.session_state["gamma"] = round(gb, 2)
            st.session_state["target_left"] = round(round(tb / 0.05) * 0.05, 2)

        st.button("Use these settings in the sidebar", on_click=apply, key="tune_apply")
        st.caption("Agreement is a step-like objective (a policy either matches a move or it doesn't), which is exactly "
                   "where a derivative-free method like Hooke-Jeeves is useful. It finds a local optimum from its start.")
    show_code('''def hooke_jeeves(func, x0, step_size=0.5, epsilon=1e-6, max_iter=1000):
    x = np.array(x0)
    n = len(x)
    delta = step_size
    iter_count = 0

    def explore(x, delta):
        for i in range(n):
            f_val = func(x)
            x[i] += delta
            if func(x) < f_val:
                continue
            x[i] -= 2 * delta
            if func(x) < f_val:
                continue
            x[i] += delta
        return x

    while delta > epsilon and iter_count < max_iter:
        iter_count += 1
        x_old = np.copy(x)
        x = explore(x, delta)
        if np.array_equal(x, x_old):
            delta /= 2
        else:
            x = x + (x - x_old)
    return x

def objective_function(x):
    return x[0]**2 + x[1]**2
print("Optimized parameters:", hooke_jeeves(objective_function, [1.0, 1.0]))''')


def page_results():
    st.title("📊 Data & results")
    headline()
    tabs = st.tabs(["Sensors", "MDP model", "Value function & CSV", "Policy check"])
    with tabs[0]:
        i = st.slider("Log reading", 0, len(df) - 1, min(1500, len(df) - 1), key="res_row")
        row = df.iloc[i]
        a, b = st.columns([1, 1])
        with a:
            theta = [E.ANGLE[k] for k in range(1, 25)] + [E.ANGLE[1]]
            r = [row[f"US{k}"] for k in range(1, 25)] + [row["US1"]]
            fig = go.Figure(go.Scatterpolar(r=r, theta=theta, mode="lines+markers", fill="toself",
                                            line=dict(color="#2EC4B6"), fillcolor="rgba(46,196,182,0.2)",
                                            text=[f"US{k}" for k in range(1, 25)] + ["US1"], hovertemplate="%{text}: %{r:.2f} m<extra></extra>"))
            for d, sensors in E.ARCS.items():
                fig.add_trace(go.Scatterpolar(r=[row[f"US{k}"] for k in sensors], theta=[E.ANGLE[k] for k in sensors], mode="markers",
                                              marker=dict(size=10, color=WB.ARC_COLORS[d]), name=d))
            fig.update_layout(polar=dict(bgcolor="#101A24", radialaxis=dict(range=[0, 5.1], gridcolor=GRIDC), angularaxis=dict(direction="clockwise", rotation=90, gridcolor=GRIDC)))
            chart(styled(fig, 430, "The 24 ultrasound readings (coloured = the four arcs)"))
        with b:
            st.markdown(f"**Logged move:** {row['Class']}  \n**State:** `{dfs.iloc[i]['State']}`  \n**Optimal move:** {best_map.get(dfs.iloc[i]['State'])}")
            table(pd.DataFrame({"Distance": E.DIST, "Sensors": [", ".join(f"US{k}" for k in E.ARCS[d]) for d in E.DIST],
                                "Metres": [round(float(row[d]), 3) for d in E.DIST]}))
            dsel = st.selectbox("Distribution by move", E.DIST, index=1, key="res_dist")
            fig = go.Figure([go.Violin(y=df.loc[df["Class"] == m, dsel], name=m.replace("-", " "), line_color=MCOL[m], box_visible=True, meanline_visible=True) for m in E.MOVES])
            chart(styled(fig, 300))
    with tabs[1]:
        st.markdown("**Levels**")
        table(pd.DataFrame([{"Distance": d, **{nm: f"{lo} – {hi} m" for nm, lo, hi in zip(E.level_names(cuts[d]), ["0"] + [f"{c:.2f}" for c in cuts[d]], [f"{c:.2f}" for c in cuts[d]] + ["5"])}} for d in used]))
        act = st.selectbox("Transitions for move", E.MOVES, key="res_move")
        a_i = E.MOVES.index(act)
        rows_ = np.where(mdp.seen[:, a_i])[0]
        if len(rows_):
            rows_ = rows_[np.argsort(-mdp.counts[rows_, a_i].sum(1))][:22]
            cols_ = np.where(mdp.P[rows_, a_i].sum(0) > 0)[0]
            fig = go.Figure(go.Heatmap(z=mdp.P[np.ix_(rows_, [a_i], cols_)][:, 0, :], x=[mdp.states[c] for c in cols_], y=[mdp.states[r] for r in rows_],
                                       colorscale=[[0, "#101A24"], [1, "#2EC4B6"]], colorbar=dict(title="P")))
            chart(styled(fig, 560, f"P(s' | s, {act}) for the most common states"))
        rdf = pd.DataFrame(np.where(mdp.seen, mdp.R, np.nan), index=mdp.states, columns=E.MOVES)
        fig = go.Figure(go.Heatmap(z=rdf.values, x=E.MOVES, y=mdp.states, colorscale=[[0, "#FF6B6B"], [0.5, "#1B2733"], [1, "#2EC4B6"]], zmid=0, colorbar=dict(title="R")))
        chart(styled(fig, max(360, 16 * len(mdp.states)), "Rewards R(s, a) (blank = move never taken in that state)"))
    with tabs[2]:
        a, b = st.columns([1.3, 1])
        with a:
            fig = go.Figure(go.Bar(x=vtab["Optimal_Value"], y=vtab["State"], orientation="h",
                                   marker_color=[MCOL[m] for m in vtab["Optimal_Action"]]))
            fig.update_yaxes(autorange="reversed", showticklabels=len(vtab) <= 45)
            chart(styled(fig, max(380, 14 * len(vtab)), "V*(s), coloured by the optimal move"))
        with b:
            d = sol["deltas"]
            fig = go.Figure(go.Scatter(x=np.arange(1, len(d) + 1), y=np.maximum(d, 1e-16), line=dict(color="#2EC4B6", width=2.5)))
            fig.add_hline(y=tol, line_dash="dash", line_color="#FF6B6B")
            fig.update_yaxes(type="log")
            chart(styled(fig, 280, "Largest change per sweep"))
            st.metric("Value iteration vs policy iteration", f"max |ΔV| = {np.abs(pi_res['V'] - sol['V']).max():.1e}",
                      f"same policy in {np.mean(pi_res['pi'] == sol['pi']):.0%} of states", delta_color="off")
        st.markdown("**optimal_value_function.csv**")
        table(vtab, height=380)
        st.download_button("⬇ Download optimal_value_function.csv", vtab.to_csv(index=False).encode(), "optimal_value_function.csv", "text/csv")
        if csv_saved:
            st.caption(f"Also saved as {CSV_OUT.name} next to app.py.")
    with tabs[3]:
        cm = pd.crosstab(dfs["Class"], dfs["State"].map(best_map)).reindex(index=E.MOVES, columns=E.MOVES, fill_value=0)
        fig = go.Figure(go.Heatmap(z=cm.values, x=E.MOVES, y=E.MOVES, text=cm.values, texttemplate="%{text}",
                                   colorscale=[[0, "#101A24"], [1, "#5AA9E6"]], colorbar=dict(title="readings")))
        fig.update_xaxes(title="optimal move"); fig.update_yaxes(title="logged move", autorange="reversed")
        chart(styled(fig, 440, f"Logged vs optimal move ({agree:.1%} agree)"))
        st.caption("The logged moves came from the robot's own controller, not from this reward, so they won't match "
                   "everywhere. Where they differ, the MDP expects a different move to pay off more under the shaping "
                   "set in the sidebar. The Hooke-Jeeves page can tune γ and the target to raise the agreement.")


ROUTES = {PAGES[0]: page_overview, PAGES[1]: page_simulator, PAGES[2]: page_vi, PAGES[3]: page_mdp,
          PAGES[4]: page_adp, PAGES[5]: page_mc, PAGES[6]: page_hj, PAGES[7]: page_results}
try:
    ROUTES[page]()
except Exception as exc:                                   # keep the app usable whatever happens on one page
    st.error(f"This page hit a problem: {exc}")
    st.exception(exc)


In [ ]:
import py_compile
for f in ("engine.py", "programs.py", "web_bridge.py", "app.py"):
    py_compile.compile(f, doraise=True)
for f in ("web/simulator.html", "web/vi_player.html", ".streamlit/config.toml", "requirements.txt"):
    assert Path(f).stat().st_size > 0, f
print("All files written and compiled without errors.")

## 6. Launch
Jupyter / VS Code: the app opens at the printed localhost link. Colab: use the printed link (Colab proxy first, public tunnel as a fallback).

In [ ]:
import subprocess, time, re, threading, urllib.request
from IPython.display import HTML, display

APP_PORT = 8502
for name in ("app_proc", "tunnel_proc"):
    old = globals().get(name)
    if old is not None:
        old.terminate()
app_proc, tunnel_proc = None, None

app_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", str(APP_PORT),
                             "--server.headless", "true", "--server.enableCORS", "false",
                             "--server.enableXsrfProtection", "false"],
                            stdout=open("wallbot.log", "w"), stderr=subprocess.STDOUT)

def is_up():
    try:
        urllib.request.urlopen(f"http://localhost:{APP_PORT}", timeout=1)
        return True
    except Exception:
        return False

for _ in range(60):
    if is_up():
        break
    time.sleep(1)

if not is_up():
    print("The app did not start. Log:\n", open("wallbot.log").read()[-2000:])
elif not IN_COLAB:
    display(HTML(f'<a href="http://localhost:{APP_PORT}" target="_blank" style="font-size:18px">🛰️ Open WallBot Lab: http://localhost:{APP_PORT}</a>'))
else:
    try:
        from google.colab.output import eval_js
        print("Colab link:", eval_js(f"google.colab.kernel.proxyPort({APP_PORT})"))
    except Exception as exc:
        print("Colab proxy unavailable:", exc)
    try:
        if not Path("cloudflared").exists():
            urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
            os.chmod("cloudflared", 0o755)
        tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{APP_PORT}", "--no-autoupdate"],
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        box = []
        threading.Thread(target=lambda: [box.append(m.group(0)) for line in tunnel_proc.stdout
                                         for m in [re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)] if m and not box],
                         daemon=True).start()
        for _ in range(45):
            if box:
                break
            time.sleep(1)
        if box:
            display(HTML(f'<a href="{box[0]}" target="_blank" style="font-size:18px">🛰️ Open WallBot Lab (public link): {box[0]}</a>'))
    except Exception as exc:
        print("Public tunnel unavailable:", exc)

### Stop the app

In [ ]:
for name in ("app_proc", "tunnel_proc"):
    proc = globals().get(name)
    if proc is not None:
        proc.terminate()
print("WallBot Lab stopped.")